# Очистка и подготовка user-course датасета для моделирования статуса

Этот ноутбук собирает рабочую основу для модельного датасета на уровне `user-course`.

Документ устроен так:

- сначала загружаются и типизируются основные таблицы LMS;
- затем проверяются ключи, связность, дубликаты и базовые аномалии;
- после этого строятся агрегаты по сырым логам до уровня `user-course`;
- в конце формируется единая витрина для следующего этапа feature engineering и обучения модели.

Важно: в этом ноутбуке не обрабатываются сырые `stats__module_*`. Вместо них используется уже подготовленная объединённая таблица `./modules/status_modules_complete.csv`, которая задаёт population на уровне модульного `user-course`, содержит фактический статус там, где он уже известен, и даёт лучшую эвристику статуса для модулей `1-3`.


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.options.display.max_columns = 200
pd.options.display.max_rows = 200


def resolve_base_dir() -> Path:
    candidates = [Path.cwd(), Path.cwd() / "hackathon"]
    for candidate in candidates:
        if (candidate / "src").exists() and (candidate / "modules").exists():
            return candidate.resolve()
    raise FileNotFoundError(
        "Не удалось найти папки ./src и ./modules относительно текущего ноутбука. "
        "Откройте ноутбук из каталога hackathon или проверьте рабочую директорию."
    )


BASE_DIR = resolve_base_dir()
SRC_DIR = BASE_DIR / "src"
MODULES_DIR = BASE_DIR / "modules"
OUTPUT_DIR = BASE_DIR / "artifacts"
OUTPUT_DIR.mkdir(exist_ok=True)

CSV_KWARGS = {
    "thousands": ",",
    "true_values": ["True"],
    "false_values": ["False"],
}


def read_typed_csv(
    path: Path,
    *,
    usecols: list[str],
    numeric_cols: dict[str, str] | None = None,
    bool_cols: list[str] | None = None,
    category_cols: list[str] | None = None,
    datetime_cols: dict[str, str] | None = None,
    extra_csv_kwargs: dict | None = None,
) -> pd.DataFrame:
    read_kwargs = dict(CSV_KWARGS)
    if extra_csv_kwargs:
        read_kwargs.update(extra_csv_kwargs)

    df = pd.read_csv(path, usecols=usecols, **read_kwargs)

    for col, fmt in (datetime_cols or {}).items():
        df[col] = pd.to_datetime(df[col], format=fmt, errors="coerce")

    for col, dtype in (numeric_cols or {}).items():
        df[col] = pd.to_numeric(df[col], errors="coerce").astype(dtype)

    for col in (bool_cols or []):
        df[col] = df[col].astype("boolean")

    for col in (category_cols or []):
        df[col] = df[col].astype("category")

    return df



def bool_sum(series: pd.Series) -> int:
    return int(series.fillna(False).astype("int8").sum())


def pct(numerator: int, denominator: int) -> float:
    return round(numerator / denominator * 100, 2) if denominator else np.nan


def fk_report(name: str, child: pd.Series, parent: pd.Series) -> dict:
    child_non_null = child.dropna()
    parent_index = pd.Index(parent.dropna().unique())
    total = int(child_non_null.shape[0])
    matched = int(child_non_null.isin(parent_index).sum())
    return {
        "check": name,
        "rows_checked": total,
        "matched_rows": matched,
        "unmatched_rows": total - matched,
        "coverage_pct": pct(matched, total),
    }




## Что именно загружаем и почему

Здесь читаются только те таблицы, которые реально помогают собрать витрину на уровне `user-course` или нужны для контроля качества связей.

Осознанные решения:

- `stats__module_*` исключены из ноутбука полностью: они обрабатываются отдельно;
- объединённый `modules/status_modules_complete.csv` используется как модульная population-таблица и источник фактического/эвристического статуса;
- загружаются только те поля, которые нужны для диагностики качества или дальнейших user-course агрегатов;
- таблицы наград здесь не используются: они не course-specific и легко создают риск временной утечки.

Ниже мы сначала подготавливаем typed DataFrame-ы, а затем строим признаки из сырых логов LMS.


In [2]:
users_df = read_typed_csv(
    SRC_DIR / "users.csv",
    usecols=[
        "id",
        "created_at",
        "updated_at",
        "sign_in_count",
        "last_sign_in_at",
        "timezone",
        "xp",
    ],
    numeric_cols={
        "id": "Int32",
        "sign_in_count": "Int16",
        "xp": "Int32",
    },
    category_cols=["timezone"],
    datetime_cols={
        "created_at": "%d %b, %Y, %H:%M",
        "updated_at": "%d %b, %Y, %H:%M",
        "last_sign_in_at": "%d %b, %Y, %H:%M",
    },
)

users_courses_df = read_typed_csv(
    SRC_DIR / "users_courses.csv",
    usecols=[
        "id",
        "user_id",
        "course_id",
        "created_at",
        "updated_at",
        "access_finished_at",
        "wk_points",
        "wk_max_points",
        "wk_max_task_count",
    ],
    numeric_cols={
        "id": "Int32",
        "user_id": "Int32",
        "course_id": "Int32",
        "wk_points": "Float32",
        "wk_max_points": "Float32",
        "wk_max_task_count": "Float32",
    },
    datetime_cols={
        "created_at": "%d %b, %Y, %H:%M",
        "updated_at": "%d %b, %Y, %H:%M",
        "access_finished_at": "%d %b, %Y",
    },
)
users_courses_df = users_courses_df.rename(columns={"id": "users_course_id"})

lessons_df = read_typed_csv(
    SRC_DIR / "lessons.csv",
    usecols=[
        "id",
        "course_id",
        "conspect_expected",
        "task_expected",
        "lesson_number",
        "wk_survival_training_expected",
        "wk_attendance_tracking_enabled",
        "wk_video_duration",
    ],
    numeric_cols={
        "id": "Int32",
        "course_id": "Int32",
        "lesson_number": "Int16",
        "wk_video_duration": "Float32",
    },
    bool_cols=[
        "conspect_expected",
        "task_expected",
        "wk_survival_training_expected",
        "wk_attendance_tracking_enabled",
    ],
)
lessons_df = lessons_df.rename(columns={"id": "lesson_id"})

groups_df = read_typed_csv(
    SRC_DIR / "groups.csv",
    usecols=[
        "id",
        "lesson_id",
        "duration",
        "group_template_id",
        "wk_actual_started_at",
        "wk_actual_finished_at",
    ],
    numeric_cols={
        "id": "Int32",
        "lesson_id": "Int32",
        "duration": "Int16",
        "group_template_id": "Int32",
    },
    datetime_cols={
        "wk_actual_started_at": "%d %b, %Y, %H:%M",
        "wk_actual_finished_at": "%d %b, %Y, %H:%M",
    },
)
groups_df = groups_df.rename(columns={"id": "group_id"})

trainings_df = read_typed_csv(
    SRC_DIR / "trainings.csv",
    usecols=["id", "lesson_id", "task_templates_count"],
    numeric_cols={
        "id": "Int32",
        "lesson_id": "Int32",
        "task_templates_count": "Int16",
    },
)
trainings_df = trainings_df.rename(columns={"id": "training_id"})

lesson_tasks_df = read_typed_csv(
    SRC_DIR / "lesson_tasks.csv",
    usecols=["id", "lesson_id", "task_id", "position", "task_required"],
    numeric_cols={
        "id": "Int32",
        "lesson_id": "Int32",
        "task_id": "Int32",
        "position": "Int16",
    },
    bool_cols=["task_required"],
)

homeworks_df = read_typed_csv(
    SRC_DIR / "homeworks.csv",
    usecols=["id", "resource_type", "resource_id"],
    numeric_cols={"id": "Int32", "resource_id": "Int32"},
    category_cols=["resource_type"],
)
homeworks_df = homeworks_df.rename(columns={"id": "homework_id"})

homework_items_df = read_typed_csv(
    SRC_DIR / "homework_items.csv",
    usecols=["id", "homework_id", "resource_id", "position"],
    numeric_cols={
        "id": "Int32",
        "homework_id": "Int32",
        "resource_id": "Int32",
        "position": "Int16",
    },
)

user_access_histories_df = read_typed_csv(
    SRC_DIR / "user_access_histories.csv",
    usecols=["users_course_id", "access_started_at", "access_expired_at", "activator_class"],
    numeric_cols={"users_course_id": "Int32"},
    category_cols=["activator_class"],
    datetime_cols={
        "access_started_at": "%d %b, %Y",
        "access_expired_at": "%d %b, %Y",
    },
)

user_lessons_df = read_typed_csv(
    SRC_DIR / "user_lessons.csv",
    usecols=[
        "id",
        "user_id",
        "lesson_id",
        "group_id",
        "video_visited",
        "video_viewed",
        "translation_visited",
        "users_course_id",
        "solved",
        "solved_tasks_count",
        "wk_points",
        "wk_solved_task_count",
    ],
    numeric_cols={
        "id": "Int32",
        "user_id": "Int32",
        "lesson_id": "Int32",
        "group_id": "Int32",
        "users_course_id": "Int32",
        "solved_tasks_count": "Int16",
        "wk_points": "Float32",
        "wk_solved_task_count": "Int16",
    },
    bool_cols=["video_visited", "video_viewed", "translation_visited", "solved"],
)
user_lessons_df = user_lessons_df.rename(columns={"id": "user_lesson_id"})

user_activity_histories_df = read_typed_csv(
    SRC_DIR / "user_activity_histories.csv",
    usecols=["user_lesson_id", "action", "created_at"],
    numeric_cols={"user_lesson_id": "Int32"},
    category_cols=["action"],
    datetime_cols={"created_at": "%Y-%m-%d %H:%M:%S"},
)

user_answers_df = read_typed_csv(
    SRC_DIR / "user_answers.csv",
    usecols=[
        "user_id",
        "task_id",
        "attempts",
        "solved",
        "points",
        "max_attempts",
        "skipped",
        "resource_type",
        "resource_id",
        "submitted_at",
        "wk_partial_answer",
        "async_check_status",
    ],
    numeric_cols={
        "user_id": "Int32",
        "task_id": "Int32",
        "attempts": "Int16",
        "points": "Float32",
        "max_attempts": "Int16",
        "resource_id": "Int32",
        "async_check_status": "Int8",
    },
    bool_cols=["solved", "skipped", "wk_partial_answer"],
    category_cols=["resource_type"],
    datetime_cols={"submitted_at": "%Y-%m-%d %H:%M:%S"},
)

user_trainings_df = read_typed_csv(
    SRC_DIR / "user_trainings.csv",
    usecols=[
        "user_id",
        "training_id",
        "solved_tasks_count",
        "earned_points",
        "submitted_answers_count",
        "started_at",
        "finished_at",
        "attempts",
        "mark",
    ],
    numeric_cols={
        "user_id": "Int32",
        "training_id": "Int32",
        "solved_tasks_count": "Int16",
        "earned_points": "Float32",
        "submitted_answers_count": "Int16",
        "attempts": "Int16",
        "mark": "Float32",
    },
    datetime_cols={
        "started_at": "%d %b, %Y, %H:%M",
        "finished_at": "%d %b, %Y, %H:%M",
    },
)

wk_users_courses_actions_df = read_typed_csv(
    SRC_DIR / "wk_users_courses_actions.csv",
    usecols=["user_id", "users_course_id", "action", "created_at", "updated_at", "lesson_id"],
    numeric_cols={"user_id": "Int32", "users_course_id": "Int32", "lesson_id": "Int32"},
    category_cols=["action"],
    datetime_cols={
        "created_at": "%Y-%m-%d %H:%M:%S",
        "updated_at": "%Y-%m-%d %H:%M:%S",
    },
)

wk_media_view_sessions_df = read_typed_csv(
    SRC_DIR / "wk_media_view_sessions.csv",
    usecols=["resource_type", "resource_id", "viewer_id", "segments_total", "viewed_segments_count", "started_at", "kind"],
    numeric_cols={
        "resource_id": "Int32",
        "viewer_id": "Int32",
        "segments_total": "Int16",
        "viewed_segments_count": "Int16",
    },
    category_cols=["resource_type", "kind"],
    datetime_cols={"started_at": "%d %b, %Y, %H:%M"},
)

modules_status_df = read_typed_csv(
    MODULES_DIR / "status_modules_complete.csv",
    usecols=[
        "module",
        "user_id",
        "club_name",
        "teacher_id",
        "course_id",
        "cohort_id",
        "level_bin",
        "enrollment_date",
        "status_actual",
        "status_by_best_heuristic",
    ],
    numeric_cols={
        "module": "Int8",
        "user_id": "Int32",
        "teacher_id": "Int32",
        "course_id": "Int32",
        "cohort_id": "Int32",
        "level_bin": "Int8",
    },
    category_cols=["club_name", "status_actual", "status_by_best_heuristic"],
    datetime_cols={"enrollment_date": "%Y-%m-%d"},
)

dataframes = {
    "users_df": users_df,
    "users_courses_df": users_courses_df,
    "lessons_df": lessons_df,
    "groups_df": groups_df,
    "trainings_df": trainings_df,
    "lesson_tasks_df": lesson_tasks_df,
    "homeworks_df": homeworks_df,
    "homework_items_df": homework_items_df,
    "user_access_histories_df": user_access_histories_df,
    "user_lessons_df": user_lessons_df,
    "user_activity_histories_df": user_activity_histories_df,
    "user_answers_df": user_answers_df,
    "user_trainings_df": user_trainings_df,
    "wk_users_courses_actions_df": wk_users_courses_actions_df,
    "wk_media_view_sessions_df": wk_media_view_sessions_df,
    "modules_status_df": modules_status_df,
}

inventory_df = pd.DataFrame(
    {
        "dataframe": list(dataframes.keys()),
        "rows": [df.shape[0] for df in dataframes.values()],
        "cols": [df.shape[1] for df in dataframes.values()],
    }
)

display(inventory_df.sort_values(["rows", "cols"], ascending=[False, False]).reset_index(drop=True))
display(modules_status_df.head())


,dataframe,rows,cols
0,user_answers_df,15176182,12
1,wk_users_courses_actions_df,12909207,6
2,user_lessons_df,3070664,12
3,user_activity_histories_df,3031137,3
4,wk_media_view_sessions_df,852358,7
5,user_access_histories_df,667124,4
6,user_trainings_df,427628,9
7,users_courses_df,290835,9
8,users_df,95395,7
9,lesson_tasks_df,29544,5


,module,user_id,club_name,teacher_id,course_id,cohort_id,level_bin,enrollment_date,status_actual,status_by_best_heuristic
0,1,701741,Python. Начальный уровень. Онлайн. 1 модуль,699869,1029,1149,0,2025-09-19,Завершил,Completed
1,1,737977,Python. Начальный уровень. Онлайн. 1 модуль,730341,1029,1216,0,2025-11-05,Завершил,Completed
2,1,722851,Python. Базовый уровень. Онлайн. 1 модуль,730208,1030,1186,1,2025-10-21,Отчислен,Dropped
3,1,709724,Python. Базовый уровень. Онлайн. 1 модуль,700089,1030,1108,1,2025-09-23,Завершил,Completed
4,1,717763,Python. Базовый уровень. Онлайн. 1 модуль,718519,1030,1055,1,2025-10-09,Отчислен,Dropped


## Первичная проверка качества

На этом шаге важно не просто посмотреть `shape`, а зафиксировать, что именно можно использовать как надёжную связь для user-course агрегатов.

Проверяем четыре группы вопросов:

- грубые дубликаты и нарушения ключей;
- покрытие FK между таблицами;
- временные аномалии и базовые sanity-checks;
- качество сигналов просмотра, событий внутри урока и стыковку `modules/status_modules_complete.csv` с `users_courses`.


In [3]:
duplicate_rows_df = pd.DataFrame(
    {
        "dataframe": list(dataframes.keys()),
        "duplicate_rows": [int(df.duplicated().sum()) for df in dataframes.values()],
        "rows": [df.shape[0] for df in dataframes.values()],
    }
)
duplicate_rows_df["duplicate_share_pct"] = (
    duplicate_rows_df["duplicate_rows"] / duplicate_rows_df["rows"] * 100
).round(2)

key_uniqueness_df = pd.DataFrame(
    [
        {"table": "users_df", "key": "user_id", "duplicates": int(users_df.duplicated(["id"]).sum())},
        {"table": "users_courses_df", "key": "users_course_id", "duplicates": int(users_courses_df.duplicated(["users_course_id"]).sum())},
        {"table": "users_courses_df", "key": "(user_id, course_id)", "duplicates": int(users_courses_df.duplicated(["user_id", "course_id"]).sum())},
        {"table": "lessons_df", "key": "lesson_id", "duplicates": int(lessons_df.duplicated(["lesson_id"]).sum())},
        {"table": "groups_df", "key": "group_id", "duplicates": int(groups_df.duplicated(["group_id"]).sum())},
        {"table": "trainings_df", "key": "training_id", "duplicates": int(trainings_df.duplicated(["training_id"]).sum())},
        {"table": "lesson_tasks_df", "key": "id", "duplicates": int(lesson_tasks_df.duplicated(["id"]).sum())},
        {"table": "homeworks_df", "key": "homework_id", "duplicates": int(homeworks_df.duplicated(["homework_id"]).sum())},
        {"table": "homework_items_df", "key": "id", "duplicates": int(homework_items_df.duplicated(["id"]).sum())},
        {"table": "user_lessons_df", "key": "user_lesson_id", "duplicates": int(user_lessons_df.duplicated(["user_lesson_id"]).sum())},
        {"table": "user_lessons_df", "key": "(user_id, users_course_id, lesson_id)", "duplicates": int(user_lessons_df.duplicated(["user_id", "users_course_id", "lesson_id"]).sum())},
        {"table": "user_activity_histories_df", "key": "(user_lesson_id, action, created_at)", "duplicates": int(user_activity_histories_df.duplicated(["user_lesson_id", "action", "created_at"]).sum())},
        {"table": "modules_status_df", "key": "(module, user_id, course_id)", "duplicates": int(modules_status_df.duplicated(["module", "user_id", "course_id"]).sum())},
    ]
)

fk_checks_df = pd.DataFrame(
    [
        fk_report("users_courses.user_id -> users.id", users_courses_df["user_id"], users_df["id"]),
        fk_report("user_lessons.user_id -> users.id", user_lessons_df["user_id"], users_df["id"]),
        fk_report("user_lessons.users_course_id -> users_courses.users_course_id", user_lessons_df["users_course_id"], users_courses_df["users_course_id"]),
        fk_report("user_lessons.lesson_id -> lessons.lesson_id", user_lessons_df["lesson_id"], lessons_df["lesson_id"]),
        fk_report("user_lessons.group_id -> groups.group_id", user_lessons_df["group_id"], groups_df["group_id"]),
        fk_report("user_activity_histories.user_lesson_id -> user_lessons.user_lesson_id", user_activity_histories_df["user_lesson_id"], user_lessons_df["user_lesson_id"]),
        fk_report("groups.lesson_id -> lessons.lesson_id", groups_df["lesson_id"], lessons_df["lesson_id"]),
        fk_report("trainings.lesson_id -> lessons.lesson_id", trainings_df["lesson_id"], lessons_df["lesson_id"]),
        fk_report("user_trainings.training_id -> trainings.training_id", user_trainings_df["training_id"], trainings_df["training_id"]),
        fk_report("wk_media_view_sessions.viewer_id -> users.id", wk_media_view_sessions_df["viewer_id"], users_df["id"]),
        fk_report(
            "wk_media_view_sessions.resource_id[Group] -> groups.group_id",
            wk_media_view_sessions_df.loc[wk_media_view_sessions_df["resource_type"].eq("Group"), "resource_id"],
            groups_df["group_id"],
        ),
        fk_report(
            "wk_media_view_sessions.resource_id[Lesson] -> lessons.lesson_id",
            wk_media_view_sessions_df.loc[wk_media_view_sessions_df["resource_type"].eq("Lesson"), "resource_id"],
            lessons_df["lesson_id"],
        ),
        fk_report("user_answers.user_id -> users.id", user_answers_df["user_id"], users_df["id"]),
        fk_report("modules_status.user_id -> users.id", modules_status_df["user_id"], users_df["id"]),
        fk_report("modules_status.course_id -> lessons.course_id", modules_status_df["course_id"], lessons_df["course_id"]),
        fk_report("modules_status.cohort_id -> groups.group_template_id", modules_status_df["cohort_id"], groups_df["group_template_id"]),
    ]
)

temporal_checks_df = pd.DataFrame(
    [
        {"table": "users_df", "check": "updated_at < created_at", "bad_rows": int((users_df["updated_at"] < users_df["created_at"]).sum())},
        {"table": "users_courses_df", "check": "updated_at < created_at", "bad_rows": int((users_courses_df["updated_at"] < users_courses_df["created_at"]).sum())},
        {"table": "users_courses_df", "check": "wk_points > wk_max_points", "bad_rows": int((users_courses_df["wk_points"] > users_courses_df["wk_max_points"]).sum())},
        {"table": "groups_df", "check": "wk_actual_finished_at < wk_actual_started_at", "bad_rows": int((groups_df["wk_actual_finished_at"] < groups_df["wk_actual_started_at"]).sum())},
        {"table": "lessons_df", "check": "lesson_number <= 0", "bad_rows": int((lessons_df["lesson_number"] <= 0).sum())},
        {"table": "lesson_tasks_df", "check": "position <= 0", "bad_rows": int((lesson_tasks_df["position"] <= 0).sum())},
        {"table": "homework_items_df", "check": "position <= 0", "bad_rows": int((homework_items_df["position"] <= 0).sum())},
        {"table": "user_answers_df", "check": "attempts > max_attempts", "bad_rows": int((user_answers_df["attempts"] > user_answers_df["max_attempts"]).sum())},
        {"table": "user_trainings_df", "check": "finished_at < started_at", "bad_rows": int((user_trainings_df["finished_at"] < user_trainings_df["started_at"]).sum())},
        {"table": "wk_users_courses_actions_df", "check": "updated_at < created_at", "bad_rows": int((wk_users_courses_actions_df["updated_at"] < wk_users_courses_actions_df["created_at"]).sum())},
        {"table": "wk_media_view_sessions_df", "check": "viewed_segments_count > segments_total", "bad_rows": int((wk_media_view_sessions_df["viewed_segments_count"] > wk_media_view_sessions_df["segments_total"]).sum())},
    ]
)

video_signal_df = pd.DataFrame(
    [
        {"metric": "rows_video_visited_true", "value": int(user_lessons_df["video_visited"].fillna(False).sum())},
        {"metric": "rows_video_viewed_true", "value": int(user_lessons_df["video_viewed"].fillna(False).sum())},
        {"metric": "video_visited_true_and_video_viewed_false", "value": int(((user_lessons_df["video_visited"] == True) & (user_lessons_df["video_viewed"] == False)).sum())},
        {"metric": "video_viewed_true_and_video_visited_false", "value": int(((user_lessons_df["video_viewed"] == True) & (user_lessons_df["video_visited"] == False)).sum())},
    ]
)

modules_backbone_df = modules_status_df.merge(
    users_courses_df[["users_course_id", "user_id", "course_id"]],
    on=["user_id", "course_id"],
    how="left",
)
module_users_course_coverage_df = (
    modules_backbone_df
    .groupby("module", dropna=False)
    .agg(
        rows=("module", "size"),
        matched_users_course=("users_course_id", lambda s: int(s.notna().sum())),
        unmatched_users_course=("users_course_id", lambda s: int(s.isna().sum())),
    )
    .reset_index()
)
module_users_course_coverage_df["matched_pct"] = module_users_course_coverage_df.apply(
    lambda row: pct(int(row["matched_users_course"]), int(row["rows"])),
    axis=1,
)

display(duplicate_rows_df.sort_values("duplicate_rows", ascending=False).reset_index(drop=True))
display(key_uniqueness_df)
display(fk_checks_df.sort_values("coverage_pct", ascending=True).reset_index(drop=True))
display(temporal_checks_df.sort_values(["bad_rows", "table"], ascending=[False, True]).reset_index(drop=True))
display(video_signal_df)
display(module_users_course_coverage_df)


,dataframe,duplicate_rows,rows,duplicate_share_pct
0,wk_users_courses_actions_df,5491157,12909207,42.54
1,user_answers_df,4950084,15176182,32.62
2,user_access_histories_df,355372,667124,53.27
3,user_activity_histories_df,33312,3031137,1.10
4,wk_media_view_sessions_df,3558,852358,0.42
5,users_df,0,95395,0.00
6,users_courses_df,0,290835,0.00
7,lessons_df,0,3369,0.00
8,groups_df,0,13076,0.00
9,trainings_df,0,410,0.00


,table,key,duplicates
0,users_df,user_id,0
1,users_courses_df,users_course_id,0
2,users_courses_df,"(user_id, course_id)",0
3,lessons_df,lesson_id,0
4,groups_df,group_id,0
5,trainings_df,training_id,0
6,lesson_tasks_df,id,0
7,homeworks_df,homework_id,0
8,homework_items_df,id,0
9,user_lessons_df,user_lesson_id,0


,check,rows_checked,matched_rows,unmatched_rows,coverage_pct
0,trainings.lesson_id -> lessons.lesson_id,256,252,4,98.44
1,user_activity_histories.user_lesson_id -> user...,3031137,3007406,23731,99.22
2,user_lessons.lesson_id -> lessons.lesson_id,3070664,3062469,8195,99.73
3,user_lessons.group_id -> groups.group_id,3070664,3062469,8195,99.73
4,groups.lesson_id -> lessons.lesson_id,13076,13068,8,99.94
5,users_courses.user_id -> users.id,290835,290835,0,100.00
6,user_lessons.user_id -> users.id,3070664,3070615,49,100.00
7,user_lessons.users_course_id -> users_courses....,3070664,3070524,140,100.00
8,user_trainings.training_id -> trainings.traini...,427628,427628,0,100.00
9,wk_media_view_sessions.viewer_id -> users.id,852358,852358,0,100.00


,table,check,bad_rows
0,user_answers_df,attempts > max_attempts,190
1,users_courses_df,wk_points > wk_max_points,179
2,wk_users_courses_actions_df,updated_at < created_at,1
3,groups_df,wk_actual_finished_at < wk_actual_started_at,0
4,homework_items_df,position <= 0,0
5,lesson_tasks_df,position <= 0,0
6,lessons_df,lesson_number <= 0,0
7,user_trainings_df,finished_at < started_at,0
8,users_courses_df,updated_at < created_at,0
9,users_df,updated_at < created_at,0


,metric,value
0,rows_video_visited_true,2471536
1,rows_video_viewed_true,99650
2,video_visited_true_and_video_viewed_false,2371889
3,video_viewed_true_and_video_visited_false,3


,module,rows,matched_users_course,unmatched_users_course,matched_pct
0,1,2972,2972,0,100.0
1,2,1955,1955,0,100.0
2,3,1785,1785,0,100.0
3,4,1689,0,1689,0.0


## Интерпретация промежуточных результатов

Здесь важно зафиксировать несколько практических выводов, которые будут влиять на итоговую витрину.

1. `video_viewed` действительно нужен.
   В текущих данных огромное число строк имеет `video_visited=True`, но `video_viewed=False`. Это означает, что одного признака захода в видео недостаточно: для модели важнее отделять факт открытия плеера от фактического просмотра.

2. `user_activity_histories` можно связать с `user_lessons` по `user_lesson_id`.
   Это позволяет использовать события внутри урока не только для ручной проверки, но и для построения user-course агрегатов. При этом покрытие по `course_id` всё равно чуть ниже 100%, потому что часть `user_lessons` не стыкуется с `lessons`, и это ограничение надо помнить при интерпретации признаков.

3. Для модулей 1, 2 и 3 записи из `modules/status_modules_complete.csv` хорошо стыкуются с `users_courses`, а для модуля 4 такого совпадения нет.
   Поэтому итоговую витрину нельзя строить только через `users_course_id`.
   В этом документе используется смешанный подход: где связь с `users_courses` есть, берём её напрямую; где её нет, восстанавливаем принадлежность к курсу через уроки, группы, тренинги и типы ресурсов в логах.


## Удаление точных дублей перед сборкой витрины

На этом шаге удаляются дубли событий в тех таблицах, где они завышают агрегаты.

В том числе дедуплицируется `user_activity_histories_df` по ключу `(user_lesson_id, action, created_at)`: именно в этой таблице точные повторы особенно заметно завышают activity-агрегаты.


In [4]:
def drop_exact_duplicates(df: pd.DataFrame, subset: list[str], dataframe_name: str) -> tuple[pd.DataFrame, dict]:
    duplicate_rows = int(df.duplicated(subset=subset).sum())
    deduped_df = df.drop_duplicates(subset=subset).copy()
    return deduped_df, {
        "dataframe": dataframe_name,
        "rows_before": int(df.shape[0]),
        "duplicate_rows_removed": duplicate_rows,
        "rows_after": int(deduped_df.shape[0]),
        "dedup_subset": ", ".join(subset),
    }


user_activity_histories_df, user_activity_dedup_report = drop_exact_duplicates(
    user_activity_histories_df,
    ["user_lesson_id", "action", "created_at"],
    "user_activity_histories_df",
)
wk_users_courses_actions_df, wk_users_courses_actions_dedup_report = drop_exact_duplicates(
    wk_users_courses_actions_df,
    ["user_id", "users_course_id", "action", "created_at", "updated_at", "lesson_id"],
    "wk_users_courses_actions_df",
)
wk_media_view_sessions_df, wk_media_view_sessions_dedup_report = drop_exact_duplicates(
    wk_media_view_sessions_df,
    ["resource_type", "resource_id", "viewer_id", "segments_total", "viewed_segments_count", "started_at", "kind"],
    "wk_media_view_sessions_df",
)
user_answers_df, user_answers_dedup_report = drop_exact_duplicates(
    user_answers_df,
    [
        "user_id",
        "task_id",
        "attempts",
        "solved",
        "points",
        "max_attempts",
        "skipped",
        "resource_type",
        "resource_id",
        "submitted_at",
        "wk_partial_answer",
        "async_check_status",
    ],
    "user_answers_df",
)
user_access_histories_df, user_access_histories_dedup_report = drop_exact_duplicates(
    user_access_histories_df,
    ["users_course_id", "access_started_at", "access_expired_at", "activator_class"],
    "user_access_histories_df",
)

deduplication_df = pd.DataFrame(
    [
        user_activity_dedup_report,
        wk_users_courses_actions_dedup_report,
        wk_media_view_sessions_dedup_report,
        user_answers_dedup_report,
        user_access_histories_dedup_report,
    ]
)
deduplication_df["removed_share_pct"] = deduplication_df.apply(
    lambda row: pct(int(row["duplicate_rows_removed"]), int(row["rows_before"])),
    axis=1,
)

dataframes.update(
    {
        "user_activity_histories_df": user_activity_histories_df,
        "wk_users_courses_actions_df": wk_users_courses_actions_df,
        "wk_media_view_sessions_df": wk_media_view_sessions_df,
        "user_answers_df": user_answers_df,
        "user_access_histories_df": user_access_histories_df,
    }
)

activity_probe_df = (
    user_activity_histories_df
    .merge(user_lessons_df[["user_lesson_id", "user_id", "lesson_id", "users_course_id"]], on="user_lesson_id", how="left")
    .merge(lessons_df[["lesson_id", "course_id"]], on="lesson_id", how="left")
)
activity_quality_df = pd.DataFrame(
    [
        {"metric": "user_activity rows with matched user_lesson", "value": int(activity_probe_df["user_id"].notna().sum())},
        {"metric": "user_activity rows with matched course_id", "value": int(activity_probe_df["course_id"].notna().sum())},
        {"metric": "user_lessons rows with any activity", "value": int(user_lessons_df["user_lesson_id"].isin(user_activity_histories_df["user_lesson_id"]).sum())},
        {"metric": "user_activity distinct actions", "value": int(user_activity_histories_df["action"].nunique(dropna=True))},
    ]
)
activity_actions_df = (
    user_activity_histories_df["action"]
    .astype("string")
    .value_counts(dropna=False)
    .rename_axis("action")
    .reset_index(name="rows")
)

print("Точные дубли удалены, включая user_activity_histories_df по ключу (user_lesson_id, action, created_at).")
display(deduplication_df)
display(activity_quality_df)
display(activity_actions_df)


Точные дубли удалены, включая user_activity_histories_df по ключу (user_lesson_id, action, created_at).


,dataframe,rows_before,duplicate_rows_removed,rows_after,dedup_subset,removed_share_pct
0,user_activity_histories_df,3031137,33312,2997825,"user_lesson_id, action, created_at",1.10
1,wk_users_courses_actions_df,12909207,5491157,7418050,"user_id, users_course_id, action, created_at, ...",42.54
2,wk_media_view_sessions_df,852358,3558,848800,"resource_type, resource_id, viewer_id, segment...",0.42
3,user_answers_df,15176182,4950084,10226098,"user_id, task_id, attempts, solved, points, ma...",32.62
4,user_access_histories_df,667124,355372,311752,"users_course_id, access_started_at, access_exp...",53.27


,metric,value
0,user_activity rows with matched user_lesson,2974403
1,user_activity rows with matched course_id,2966185
2,user_lessons rows with any activity,2525447
3,user_activity distinct actions,3


,action,rows
0,visit_video,2628514
1,show_conspect,267738
2,visit_translation,101573


## Сборка общей витрины и проверка ее качества

1. Сначала для разных сущностей восстанавливается `course_id`.
   Для этого используются цепочки `lesson -> course`, `group -> lesson -> course`, `training -> lesson -> course`, а также доступные связи для домашних заданий.
   Это нужно потому, что далеко не все сырые события сразу лежат на уровне курса.

2. Затем собираются признаки структуры самого курса.
   Считаются базовые характеристики курса, одинаковые для всех пользователей внутри одного `course_id`: суммарная и средняя длительность видео по урокам, а также признаки по `lesson_tasks`.

3. После этого подготавливаются user-level и users-course-level признаки.
   Из `users` берутся базовые характеристики пользователя, а из `users_courses` — признаки состояния прохождения курса, доступов и накопленных баллов.

4. Отдельно агрегируется `user_lessons`.
   Здесь строятся признаки по урокам внутри курса: число уроков, групп, просмотров, решённых уроков, суммарные баллы и т.д.

5. Далее агрегируется `user_activity_histories`.
   События внутри урока сначала связываются с `user_lessons`, затем через урок восстанавливается курс, после чего считаются признаки активности: число событий, число затронутых уроков, число дней активности и распределение по типам действий.

6. Затем агрегируются `wk_users_courses_actions`.
   Это уже course-level лог действий: он даёт число событий по курсу, активные дни, временные границы активности и частоты конкретных action-типов.

7. После этого агрегируются просмотры медиа из `wk_media_view_sessions`.
   Для них восстанавливается курс через урок или группу, а затем считаются признаки глубины просмотра: число сессий, средняя доля просмотра, максимальная доля, число просмотров на 80% и полных просмотров.

8. Следующим шагом агрегируются `user_trainings`.
   Это даёт признаки по тренингам: число прохождений, число решённых задач, баллы, попытки, оценки и временные границы активности.

9. Затем агрегируются `user_answers`.
   Перед этим для ответов восстанавливается `course_id` отдельно для ресурсов типа `Lesson`, `Training` и `Homework`.
   После этого считаются course-level признаки по ответам: число задач, успешных решений, пропусков, partial answers, баллов, попыток и границы по времени.

10. Когда все сырые источники агрегированы, формируется population-таблица.
    Она задаётся через `modules/status_modules_complete.csv`: одна строка на один объект `module + user_id + course_id`.
    На этом же шаге в population появляются два бинарных таргета: фактический `actual_target` и эвристический `heuristic_target` для модулей `1-3`.

11. На следующем шаге все агрегаты последовательно присоединяются к population-таблице.
    В результате сначала получается сырая витрина `raw_modeling_dataset_df` на уровне `user-course`.

12. После первичной сборки выполняется дополнительный QC-блок.
    В нём проверяются полностью пустые, константные и почти константные признаки, а также отдельно чинятся поля, которые можно восстановить из raw-источников.

13. На этом же этапе из итоговой витрины автоматически удаляются полностью пустые и константные признаки.
    Почти константные поля остаются в QC-отчёте для отдельного решения, а не удаляются автоматически.

14. В самом конце дополнительно считаются таблицы покрытия и пропусков уже для очищенной витрины.
    Таблица пропусков при этом строится без `module == 4`, чтобы известная особенность этого модуля не искажала общую картину по остальным данным.


In [5]:
lesson_course_df = lessons_df[["lesson_id", "course_id"]].copy()

group_course_df = groups_df[["group_id", "lesson_id"]].merge(
    lesson_course_df[["lesson_id", "course_id"]],
    on="lesson_id",
    how="left",
)

training_course_df = trainings_df[["training_id", "lesson_id", "task_templates_count"]].merge(
    lesson_course_df[["lesson_id", "course_id"]],
    on="lesson_id",
    how="left",
)

homework_course_df = (
    homeworks_df.loc[homeworks_df["resource_type"].eq("Lesson"), ["homework_id", "resource_id"]]
    .rename(columns={"resource_id": "lesson_id"})
    .merge(lesson_course_df[["lesson_id", "course_id"]], on="lesson_id", how="left")
)

homework_item_course_df = homework_items_df[["homework_id", "resource_id"]].merge(
    homework_course_df[["homework_id", "course_id"]],
    on="homework_id",
    how="left",
)

course_mapping_inventory_df = pd.DataFrame(
    [
        {"mapping": "groups -> course_id через lessons", "rows": int(group_course_df.shape[0]), "mapped_rows": int(group_course_df["course_id"].notna().sum())},
        {"mapping": "trainings -> course_id через lessons", "rows": int(training_course_df.shape[0]), "mapped_rows": int(training_course_df["course_id"].notna().sum())},
        {"mapping": "homeworks[Lesson] -> course_id через lessons", "rows": int(homework_course_df.shape[0]), "mapped_rows": int(homework_course_df["course_id"].notna().sum())},
        {"mapping": "homework_items -> course_id через homework_id", "rows": int(homework_item_course_df.shape[0]), "mapped_rows": int(homework_item_course_df["course_id"].notna().sum())},
    ]
)
course_mapping_inventory_df["coverage_pct"] = course_mapping_inventory_df.apply(
    lambda row: pct(int(row["mapped_rows"]), int(row["rows"])),
    axis=1,
)

course_structure_df = (
    lessons_df.groupby("course_id", observed=True)
    .agg(
        course_video_duration_total=("wk_video_duration", "sum"),
        course_video_duration_mean=("wk_video_duration", "mean"),
    )
    .reset_index()
)

course_tasks_df = (
    lesson_tasks_df.merge(lesson_course_df[["lesson_id", "course_id"]], on="lesson_id", how="left")
    .groupby("course_id", dropna=False, observed=True)
    .agg(
        course_lesson_tasks_total=("task_id", "size"),
        course_lesson_tasks_unique=("task_id", "nunique"),
        course_required_tasks_total=("task_required", bool_sum),
    )
    .reset_index()
)

course_features_df = (
    course_structure_df
    .merge(course_tasks_df, on="course_id", how="left")
)

user_profile_df = users_df.rename(
    columns={
        "id": "user_id",
        "created_at": "user_created_at",
        "updated_at": "user_updated_at",
        "last_sign_in_at": "user_last_sign_in_at",
    }
)

users_courses_feature_df = users_courses_df.copy()
users_courses_feature_df["course_points_ratio"] = (
    users_courses_feature_df["wk_points"]
    / users_courses_feature_df["wk_max_points"].replace({0: np.nan})
)

access_course_df = user_access_histories_df.merge(
    users_courses_feature_df[["users_course_id", "user_id", "course_id"]],
    on="users_course_id",
    how="left",
)
access_course_df["access_days"] = (
    access_course_df["access_expired_at"] - access_course_df["access_started_at"]
).dt.days
access_agg_df = (
    access_course_df.groupby(["user_id", "course_id"], observed=True)
    .agg(
        access_periods_count=("users_course_id", "size"),
        access_days_total=("access_days", "sum"),
        access_started_at_first=("access_started_at", "min"),
        access_expired_at_last=("access_expired_at", "max"),
    )
    .reset_index()
)

user_lessons_course_df = user_lessons_df.merge(
    lesson_course_df[["lesson_id", "course_id"]],
    on="lesson_id",
    how="left",
)
user_lessons_agg_df = (
    user_lessons_course_df.groupby(["user_id", "course_id"], dropna=False, observed=True)
    .agg(
        user_lessons_rows=("lesson_id", "size"),
        user_lessons_nunique=("user_lesson_id", "nunique"),
        lessons_logged_nunique=("lesson_id", "nunique"),
        groups_logged_nunique=("group_id", "nunique"),
        video_visited_lessons=("video_visited", bool_sum),
        video_viewed_lessons=("video_viewed", bool_sum),
        translation_visited_lessons=("translation_visited", bool_sum),
        solved_lessons_count=("solved", bool_sum),
        solved_tasks_total=("solved_tasks_count", "sum"),
        user_lessons_points_sum=("wk_points", "sum"),
        wk_solved_task_count_total=("wk_solved_task_count", "sum"),
    )
    .reset_index()
)
user_lessons_agg_df["video_viewed_share_by_logged_lessons"] = (
    user_lessons_agg_df["video_viewed_lessons"]
    / user_lessons_agg_df["lessons_logged_nunique"].replace({0: np.nan})
)

user_activity_course_df = user_activity_histories_df.merge(
    user_lessons_course_df[["user_lesson_id", "user_id", "lesson_id", "users_course_id", "course_id"]],
    on="user_lesson_id",
    how="left",
)
user_activity_agg_df = (
    user_activity_course_df.loc[user_activity_course_df["course_id"].notna()]
    .groupby(["user_id", "course_id"], observed=True)
    .agg(
        lesson_activity_events_total=("action", "size"),
        lesson_activity_user_lessons_nunique=("user_lesson_id", "nunique"),
        lesson_activity_days=("created_at", lambda s: s.dt.normalize().nunique()),
        lesson_activity_first_at=("created_at", "min"),
        lesson_activity_last_at=("created_at", "max"),
    )
    .reset_index()
)
activity_counts_wide_df = (
    user_activity_course_df.loc[user_activity_course_df["course_id"].notna(), ["user_id", "course_id", "action"]]
    .assign(event_count=1)
    .pivot_table(
        index=["user_id", "course_id"],
        columns="action",
        values="event_count",
        aggfunc="sum",
        fill_value=0,
    )
    .rename_axis(columns=None)
    .add_prefix("lesson_activity_count__")
    .reset_index()
)
user_activity_agg_df = user_activity_agg_df.merge(activity_counts_wide_df, on=["user_id", "course_id"], how="left")

actions_course_df = wk_users_courses_actions_df.merge(
    users_courses_feature_df[["users_course_id", "user_id", "course_id"]].rename(columns={"user_id": "user_id_from_uc"}),
    on="users_course_id",
    how="left",
)
actions_agg_df = (
    actions_course_df.loc[actions_course_df["course_id"].notna()]
    .groupby(["user_id_from_uc", "course_id"], observed=True)
    .agg(
        actions_total=("action", "size"),
        action_days=("created_at", lambda s: s.dt.normalize().nunique()),
        action_first_at=("created_at", "min"),
        action_last_at=("created_at", "max"),
    )
    .reset_index()
    .rename(columns={"user_id_from_uc": "user_id"})
)
action_counts_wide_df = (
    actions_course_df.loc[actions_course_df["course_id"].notna(), ["user_id_from_uc", "course_id", "action"]]
    .assign(event_count=1)
    .pivot_table(
        index=["user_id_from_uc", "course_id"],
        columns="action",
        values="event_count",
        aggfunc="sum",
        fill_value=0,
    )
    .rename_axis(columns=None)
    .add_prefix("action_count__")
    .reset_index()
    .rename(columns={"user_id_from_uc": "user_id"})
)
actions_agg_df = actions_agg_df.merge(action_counts_wide_df, on=["user_id", "course_id"], how="left")

media_group_df = (
    wk_media_view_sessions_df.loc[wk_media_view_sessions_df["resource_type"].eq("Group")]
    .merge(group_course_df[["group_id", "course_id"]], left_on="resource_id", right_on="group_id", how="left")
    .rename(columns={"viewer_id": "user_id"})
)
media_lesson_df = (
    wk_media_view_sessions_df.loc[wk_media_view_sessions_df["resource_type"].eq("Lesson")]
    .merge(lesson_course_df[["lesson_id", "course_id"]], left_on="resource_id", right_on="lesson_id", how="left")
    .rename(columns={"viewer_id": "user_id"})
)
media_course_df = pd.concat([media_group_df, media_lesson_df], ignore_index=True, sort=False)
media_course_df["watch_share"] = np.where(
    media_course_df["segments_total"].gt(0),
    media_course_df["viewed_segments_count"] / media_course_df["segments_total"],
    np.nan,
)
media_agg_df = (
    media_course_df.groupby(["user_id", "course_id"], dropna=False, observed=True)
    .agg(
        media_sessions_total=("resource_id", "size"),
        media_resources_nunique=("resource_id", "nunique"),
        media_watch_share_mean=("watch_share", "mean"),
        media_watch_share_max=("watch_share", "max"),
        media_sessions_80pct=("watch_share", lambda s: int((s >= 0.8).sum())),
        media_sessions_full=("watch_share", lambda s: int((s >= 1.0).sum())),
        media_started_at_first=("started_at", "min"),
        media_started_at_last=("started_at", "max"),
    )
    .reset_index()
)

user_trainings_course_df = user_trainings_df.merge(
    training_course_df[["training_id", "course_id"]],
    on="training_id",
    how="left",
)
user_trainings_agg_df = (
    user_trainings_course_df.groupby(["user_id", "course_id"], dropna=False, observed=True)
    .agg(
        trainings_rows=("training_id", "size"),
        trainings_unique_total=("training_id", "nunique"),
        training_solved_tasks_total=("solved_tasks_count", "sum"),
        training_earned_points_sum=("earned_points", "sum"),
        training_submitted_answers_sum=("submitted_answers_count", "sum"),
        training_attempts_sum=("attempts", "sum"),
        training_mark_max=("mark", "max"),
        training_started_at_first=("started_at", "min"),
        training_finished_at_last=("finished_at", "max"),
    )
    .reset_index()
)

answers_base_cols = [
    "user_id",
    "task_id",
    "attempts",
    "solved",
    "points",
    "max_attempts",
    "skipped",
    "resource_id",
    "submitted_at",
    "wk_partial_answer",
    "async_check_status",
]
answers_lesson_df = (
    user_answers_df.loc[user_answers_df["resource_type"].eq("Lesson"), answers_base_cols]
    .merge(lesson_course_df[["lesson_id", "course_id"]], left_on="resource_id", right_on="lesson_id", how="left")
)
answers_training_df = (
    user_answers_df.loc[user_answers_df["resource_type"].eq("Training"), answers_base_cols]
    .merge(training_course_df[["training_id", "course_id"]], left_on="resource_id", right_on="training_id", how="left")
)
answers_homework_df = (
    user_answers_df.loc[user_answers_df["resource_type"].eq("Homework"), answers_base_cols]
    .merge(homework_course_df[["homework_id", "course_id"]], left_on="resource_id", right_on="homework_id", how="left")
)
user_answers_course_df = pd.concat(
    [answers_lesson_df, answers_training_df, answers_homework_df],
    ignore_index=True,
    sort=False,
)
answer_mapping_coverage_df = pd.DataFrame(
    [
        {"resource_type": "Lesson", "rows": int(answers_lesson_df.shape[0]), "mapped_rows": int(answers_lesson_df["course_id"].notna().sum())},
        {"resource_type": "Training", "rows": int(answers_training_df.shape[0]), "mapped_rows": int(answers_training_df["course_id"].notna().sum())},
        {"resource_type": "Homework", "rows": int(answers_homework_df.shape[0]), "mapped_rows": int(answers_homework_df["course_id"].notna().sum())},
    ]
)
answer_mapping_coverage_df["coverage_pct"] = answer_mapping_coverage_df.apply(
    lambda row: pct(int(row["mapped_rows"]), int(row["rows"])),
    axis=1,
)
user_answers_agg_df = (
    user_answers_course_df.groupby(["user_id", "course_id"], dropna=False, observed=True)
    .agg(
        answer_rows=("task_id", "size"),
        answer_tasks_nunique=("task_id", "nunique"),
        answer_solved_count=("solved", bool_sum),
        answer_skipped_count=("skipped", bool_sum),
        answer_partial_count=("wk_partial_answer", bool_sum),
        answer_points_sum=("points", "sum"),
        answer_attempts_sum=("attempts", "sum"),
        answer_max_attempts_sum=("max_attempts", "sum"),
        answer_async_nonzero_count=("async_check_status", lambda s: int((s.fillna(0) != 0).sum())),
        answer_first_submitted_at=("submitted_at", "min"),
        answer_last_submitted_at=("submitted_at", "max"),
    )
    .reset_index()
)

# В population сохраняем два бинарных таргета: фактический и эвристический для модулей 1-3.
modeling_population_df = modules_status_df[
    [
        "module",
        "user_id",
        "course_id",
        "club_name",
        "teacher_id",
        "cohort_id",
        "level_bin",
        "enrollment_date",
        "status_actual",
        "status_by_best_heuristic",
    ]
].copy()
modeling_population_df["actual_target"] = modeling_population_df["status_actual"].map(
    {"Отчислен": 0, "Завершил": 1}
).astype("Int8")
modeling_population_df["heuristic_target"] = modeling_population_df["status_by_best_heuristic"].map(
    {"Dropped": 0, "Completed": 1}
).astype("Int8")
modeling_population_df = modeling_population_df.drop(columns=["status_actual", "status_by_best_heuristic"])

raw_modeling_dataset_df = (
    modeling_population_df
    .merge(users_courses_feature_df, on=["user_id", "course_id"], how="left")
    .merge(user_profile_df, on="user_id", how="left")
    .merge(course_features_df, on="course_id", how="left")
    .merge(access_agg_df, on=["user_id", "course_id"], how="left")
    .merge(user_lessons_agg_df, on=["user_id", "course_id"], how="left")
    .merge(user_activity_agg_df, on=["user_id", "course_id"], how="left")
    .merge(actions_agg_df, on=["user_id", "course_id"], how="left")
    .merge(media_agg_df, on=["user_id", "course_id"], how="left")
    .merge(user_trainings_agg_df, on=["user_id", "course_id"], how="left")
    .merge(user_answers_agg_df, on=["user_id", "course_id"], how="left")
)

print("Для какой доли строк удалось восстановить course_id?")
display(course_mapping_inventory_df)
display(answer_mapping_coverage_df)


Для какой доли строк удалось восстановить course_id?


,mapping,rows,mapped_rows,coverage_pct
0,groups -> course_id через lessons,13076,13068,99.94
1,trainings -> course_id через lessons,410,252,61.46
2,homeworks[Lesson] -> course_id через lessons,1207,332,27.51
3,homework_items -> course_id через homework_id,5901,997,16.90


,resource_type,rows,mapped_rows,coverage_pct
0,Lesson,6669122,6657186,99.82
1,Training,3263364,3237117,99.20
2,Homework,293612,293610,100.00


## Дополнительная очистка и финализация витрины

После первичной сборки `raw_modeling_dataset_df` переводим её в финальную `modeling_dataset_df`.

На этом шаге делаем три вещи:

1. проводим аудит качества признаков;
2. чиним те поля, которые реально можно восстановить из raw-источников;
3. автоматически удаляем полностью пустые и константные признаки, а почти константные оставляем в QC-отчёте для отдельной оценки.


In [6]:
def top_share(series: pd.Series) -> float:
    counts = series.astype("string").fillna("<NA>").value_counts(dropna=False, normalize=True)
    if counts.empty:
        return 1.0
    return float(counts.iloc[0])


def build_quality_report(df: pd.DataFrame) -> pd.DataFrame:
    label_col = "actual_target" if "actual_target" in df.columns else "target" if "target" in df.columns else None
    labeled_df = df.loc[df[label_col].notna()].copy() if label_col is not None else df.copy()
    rows = []

    for column in df.columns:
        all_series = df[column]
        labeled_series = labeled_df[column] if column in labeled_df.columns else all_series.iloc[0:0]
        rows.append(
            {
                "column": column,
                "dtype": str(all_series.dtype),
                "na_share_all": round(float(all_series.isna().mean()), 6),
                "na_share_labeled": round(float(labeled_series.isna().mean()), 6),
                "nunique_all": int(all_series.nunique(dropna=False)),
                "nunique_labeled": int(labeled_series.nunique(dropna=False)),
                "top_share_all": round(top_share(all_series), 6),
                "top_share_labeled": round(top_share(labeled_series), 6),
            }
        )

    report_df = pd.DataFrame(rows)
    report_df["flag_all_null_all"] = report_df["na_share_all"].eq(1.0)
    report_df["flag_constant_all"] = report_df["nunique_all"].eq(1)
    report_df["flag_constant_labeled"] = report_df["nunique_labeled"].eq(1)
    report_df["flag_near_constant_labeled"] = report_df["top_share_labeled"].ge(0.995)
    report_df["problem_flag"] = report_df[
        [
            "flag_all_null_all",
            "flag_constant_all",
            "flag_constant_labeled",
            "flag_near_constant_labeled",
        ]
    ].any(axis=1)
    return report_df.sort_values(["problem_flag", "na_share_all", "top_share_labeled"], ascending=[False, False, False]).reset_index(drop=True)


def build_course_duration_proxy(lessons_df: pd.DataFrame, groups_df: pd.DataFrame) -> pd.DataFrame:
    lessons_duration_df = lessons_df[["lesson_id", "course_id", "wk_video_duration"]].copy()
    groups_duration_df = groups_df[["lesson_id", "duration"]].copy()

    lesson_duration_proxy_df = (
        groups_duration_df.groupby("lesson_id", dropna=False)
        .agg(
            lesson_group_duration_median=("duration", "median"),
        )
        .reset_index()
    )

    lesson_duration_proxy_df = lessons_duration_df.merge(
        lesson_duration_proxy_df,
        on="lesson_id",
        how="left",
    )
    lesson_duration_proxy_df["lesson_duration_proxy"] = lesson_duration_proxy_df["wk_video_duration"].fillna(
        lesson_duration_proxy_df["lesson_group_duration_median"]
    )

    return (
        lesson_duration_proxy_df.groupby("course_id", dropna=False)
        .agg(
            course_video_duration_total_fixed=("lesson_duration_proxy", "sum"),
            course_video_duration_mean_fixed=("lesson_duration_proxy", "mean"),
        )
        .reset_index()
    )


manual_drop_reasons = {}


def build_drop_plan(df: pd.DataFrame, quality_report_df: pd.DataFrame) -> pd.DataFrame:
    rows = []

    for column, reason in manual_drop_reasons.items():
        if column in df.columns:
            rows.append({"column": column, "reason": reason, "source": "manual"})

    planned = {row["column"] for row in rows}
    auto_problem_df = quality_report_df.loc[
        quality_report_df["flag_all_null_all"] | quality_report_df["flag_constant_all"],
        ["column", "flag_all_null_all", "flag_constant_all"],
    ]
    for record in auto_problem_df.itertuples(index=False):
        if record.column in planned:
            continue
        reason = "Поле полностью пустое." if record.flag_all_null_all else "Поле константно во всей витрине."
        rows.append({"column": record.column, "reason": reason, "source": "auto"})

    if not rows:
        return pd.DataFrame(columns=["column", "reason", "source"])
    return pd.DataFrame(rows).drop_duplicates(subset=["column"]).sort_values("column").reset_index(drop=True)


quality_before_df = build_quality_report(raw_modeling_dataset_df)
problem_before_df = quality_before_df.loc[quality_before_df["problem_flag"]].copy()

course_duration_proxy_df = build_course_duration_proxy(lessons_df, groups_df)
repaired_modeling_dataset_df = raw_modeling_dataset_df.merge(course_duration_proxy_df, on="course_id", how="left")

total_replace_mask = (
    repaired_modeling_dataset_df["course_video_duration_total"].fillna(0).eq(0)
    & repaired_modeling_dataset_df["course_video_duration_total_fixed"].notna()
)
mean_replace_mask = (
    repaired_modeling_dataset_df["course_video_duration_mean"].isna()
    & repaired_modeling_dataset_df["course_video_duration_mean_fixed"].notna()
)

repaired_modeling_dataset_df.loc[total_replace_mask, "course_video_duration_total"] = repaired_modeling_dataset_df.loc[
    total_replace_mask, "course_video_duration_total_fixed"
]
repaired_modeling_dataset_df.loc[mean_replace_mask, "course_video_duration_mean"] = repaired_modeling_dataset_df.loc[
    mean_replace_mask, "course_video_duration_mean_fixed"
]

repair_log_df = pd.DataFrame(
    [
        {
            "column": "course_video_duration_total",
            "action": "replace_zero_with_proxy",
            "affected_rows": int(total_replace_mask.sum()),
            "reason": "Использован proxy из median(groups.duration) по lesson_id.",
        },
        {
            "column": "course_video_duration_mean",
            "action": "replace_nan_with_proxy",
            "affected_rows": int(mean_replace_mask.sum()),
            "reason": "Использован тот же proxy-duration.",
        },
    ]
)

repaired_modeling_dataset_df = repaired_modeling_dataset_df.drop(
    columns=[
        "course_video_duration_total_fixed",
        "course_video_duration_mean_fixed",
    ],
    errors="ignore",
)

quality_repaired_df = build_quality_report(repaired_modeling_dataset_df)
drop_plan_df = build_drop_plan(repaired_modeling_dataset_df, quality_repaired_df)

modeling_dataset_df = repaired_modeling_dataset_df.drop(columns=drop_plan_df["column"].tolist(), errors="ignore")
quality_after_df = build_quality_report(modeling_dataset_df)
problem_after_df = quality_after_df.loc[quality_after_df["problem_flag"]].copy()

coverage_by_module_df = (
    modeling_dataset_df.groupby("module", dropna=False)
    .agg(
        rows=("module", "size"),
        labeled_rows=("actual_target", lambda s: int(s.notna().sum())),
        users_course_rows=("users_course_id", lambda s: int(s.notna().sum())),
        access_feature_rows=("access_periods_count", lambda s: int(s.notna().sum())),
        user_lessons_feature_rows=("user_lessons_rows", lambda s: int(s.notna().sum())),
        user_activity_feature_rows=("lesson_activity_events_total", lambda s: int(s.notna().sum())),
        actions_feature_rows=("actions_total", lambda s: int(s.notna().sum())),
        media_feature_rows=("media_sessions_total", lambda s: int(s.notna().sum())),
        trainings_feature_rows=("trainings_rows", lambda s: int(s.notna().sum())),
        answers_feature_rows=("answer_rows", lambda s: int(s.notna().sum())),
    )
    .reset_index()
)
for col in [
    "labeled_rows",
    "users_course_rows",
    "access_feature_rows",
    "user_lessons_feature_rows",
    "user_activity_feature_rows",
    "actions_feature_rows",
    "media_feature_rows",
    "trainings_feature_rows",
    "answers_feature_rows",
]:
    coverage_by_module_df[col.replace("_rows", "_pct")] = coverage_by_module_df.apply(
        lambda row, src=col: pct(int(row[src]), int(row["rows"])),
        axis=1,
    )

missingness_base_df = modeling_dataset_df.loc[modeling_dataset_df["module"] != 4].copy()
missingness_df = (
    missingness_base_df.isna()
    .mean()
    .sort_values(ascending=False)
    .rename("na_share")
    .reset_index()
    .rename(columns={"index": "column"})
)

summary_df = pd.DataFrame(
    [
        {
            "rows": int(raw_modeling_dataset_df.shape[0]),
            "columns_before": int(raw_modeling_dataset_df.shape[1]),
            "columns_after": int(modeling_dataset_df.shape[1]),
            "dropped_columns": int(drop_plan_df.shape[0]),
            "quality_flags_before": int(problem_before_df.shape[0]),
            "quality_flags_after": int(problem_after_df.shape[0]),
        }
    ]
)

export_path = OUTPUT_DIR / "user_course_modeling_base.csv"
modeling_dataset_df.to_csv(export_path, index=False)

print("QC по витрине до и после очистки:")
display(summary_df)
print("Какие признаки удалось починить:")
display(repair_log_df)
print("Какие признаки удаляем из финальной витрины:")
display(drop_plan_df)
print("Проблемные признаки до очистки:")
display(problem_before_df.head(40))
print("Проблемные признаки после очистки:")
display(problem_after_df.head(20))
print("Таблица покрытия итоговой очищенной user-course витрины по модулям:")
display(coverage_by_module_df)
print("Таблица пропусков по итоговой очищенной витрине без module == 4.")
display(missingness_df.head(40))


QC по витрине до и после очистки:


,rows,columns_before,columns_after,dropped_columns,quality_flags_before,quality_flags_after
0,8401,91,91,0,2,0


Какие признаки удалось починить:


,column,action,affected_rows,reason
0,course_video_duration_total,replace_zero_with_proxy,8401,Использован proxy из median(groups.duration) п...
1,course_video_duration_mean,replace_nan_with_proxy,8401,Использован тот же proxy-duration.


Какие признаки удаляем из финальной витрины:


...

,column,dtype,na_share_all,na_share_labeled,nunique_all,nunique_labeled,top_share_all,top_share_labeled,flag_all_null_all,flag_constant_all,flag_constant_labeled,flag_near_constant_labeled,problem_flag


Таблица покрытия итоговой очищенной user-course витрины по модулям:


,module,rows,labeled_rows,users_course_rows,access_feature_rows,user_lessons_feature_rows,user_activity_feature_rows,actions_feature_rows,media_feature_rows,trainings_feature_rows,answers_feature_rows,labeled_pct,users_course_pct,access_feature_pct,user_lessons_feature_pct,user_activity_feature_pct,actions_feature_pct,media_feature_pct,trainings_feature_pct,answers_feature_pct
0,1,2972,2972,2972,2972,2494,2472,2486,2415,2116,2393,100.0,100.0,100.0,83.92,83.18,83.65,81.26,71.20,80.52
1,2,1955,1955,1955,1955,1915,1913,1913,1908,1846,1870,100.0,100.0,100.0,97.95,97.85,97.85,97.60,94.42,95.65
2,3,1785,0,1785,1785,1770,1768,1774,1748,1321,1744,0.0,100.0,100.0,99.16,99.05,99.38,97.93,74.01,97.70
3,4,1689,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.00,0.00,0.00,0.00,0.00,0.00


Таблица пропусков по итоговой очищенной витрине без module == 4.


,column,na_share
0,enrollment_date,0.557211
1,actual_target,0.265942
2,training_mark_max,0.215137
3,training_finished_at_last,0.215137
4,training_attempts_sum,0.212902
5,trainings_rows,0.212902
6,trainings_unique_total,0.212902
7,training_solved_tasks_total,0.212902
8,training_earned_points_sum,0.212902
9,training_submitted_answers_sum,0.212902


# Превью готового датасета


In [7]:

display(modeling_dataset_df.head())


,module,user_id,course_id,club_name,teacher_id,cohort_id,level_bin,enrollment_date,actual_target,heuristic_target,users_course_id,created_at,updated_at,access_finished_at,wk_points,wk_max_points,wk_max_task_count,course_points_ratio,user_created_at,user_updated_at,sign_in_count,user_last_sign_in_at,timezone,xp,course_video_duration_total,course_video_duration_mean,course_lesson_tasks_total,course_lesson_tasks_unique,course_required_tasks_total,access_periods_count,access_days_total,access_started_at_first,access_expired_at_last,user_lessons_rows,user_lessons_nunique,lessons_logged_nunique,groups_logged_nunique,video_visited_lessons,video_viewed_lessons,translation_visited_lessons,solved_lessons_count,solved_tasks_total,user_lessons_points_sum,wk_solved_task_count_total,video_viewed_share_by_logged_lessons,lesson_activity_events_total,lesson_activity_user_lessons_nunique,lesson_activity_days,lesson_activity_first_at,lesson_activity_last_at,lesson_activity_count__show_conspect,lesson_activity_count__visit_translation,lesson_activity_count__visit_video,actions_total,action_days,action_first_at,action_last_at,action_count__scratch_playground_visited,action_count__start_training,action_count__user_answer,action_count__visit_preparation_material,action_count__visit_translation,action_count__visit_video,media_sessions_total,media_resources_nunique,media_watch_share_mean,media_watch_share_max,media_sessions_80pct,media_sessions_full,media_started_at_first,media_started_at_last,trainings_rows,trainings_unique_total,training_solved_tasks_total,training_earned_points_sum,training_submitted_answers_sum,training_attempts_sum,training_mark_max,training_started_at_first,training_finished_at_last,answer_rows,answer_tasks_nunique,answer_solved_count,answer_skipped_count,answer_partial_count,answer_points_sum,answer_attempts_sum,answer_max_attempts_sum,answer_async_nonzero_count,answer_first_submitted_at,answer_last_submitted_at
0,1,701741,1029,Python. Начальный уровень. Онлайн. 1 модуль,699869,1149,0,2025-09-19,1,1,588595,2025-10-28 17:22:00,2025-12-05 12:20:00,2026-04-28,138.0,140.0,153.0,0.985714,2025-09-08 02:41:00,2026-03-27 09:25:00,13,2026-03-25 08:24:00,Asia/Krasnoyarsk,649,1037.0,45.086956,110,110.0,110.0,1,182.0,2025-10-28,2026-04-28,23,23.0,23.0,23.0,20.0,13.0,0.0,23.0,153,138.0,153,0.565217,24.0,20.0,1.0,2025-12-05 10:37:00,2025-12-05 11:35:00,0.0,0.0,24.0,51.0,1.0,2025-12-05 10:36:00,2025-12-05 12:16:00,0.0,3.0,35.0,0.0,0.0,13.0,23,20.0,0.869565,1.0,20.0,17.0,2025-12-05 07:38:00,2025-12-05 08:34:00,3,3.0,43,30.0,43,3,5.0,2025-12-05 11:10:00,2025-12-05 12:20:00,117,117.0,116.0,0.0,0.0,116.0,117,117,9,2025-12-05 10:36:00,2025-12-05 12:19:00
1,1,737977,1029,Python. Начальный уровень. Онлайн. 1 модуль,730341,1216,0,2025-11-05,1,1,623160,2025-11-13 16:40:00,2025-11-29 13:54:00,2026-05-13,118.5,140.0,153.0,0.846429,2025-11-12 10:16:00,2026-03-25 10:14:00,59,2026-03-24 21:14:00,Asia/Yekaterinburg,1094,1037.0,45.086956,110,110.0,110.0,1,181.0,2025-11-13,2026-05-13,23,23.0,23.0,23.0,13.0,10.0,19.0,23.0,153,118.5,153,0.434783,67.0,20.0,12.0,2025-11-14 12:27:00,2025-11-29 13:31:00,0.0,39.0,28.0,123.0,12.0,2025-11-14 12:27:00,2025-11-29 13:32:00,0.0,3.0,58.0,0.0,39.0,23.0,30,20.0,0.65277,1.0,14.0,9.0,2025-11-14 09:28:00,2025-11-28 01:29:00,3,3.0,43,27.0,43,3,5.0,2025-11-24 13:29:00,2025-11-29 13:54:00,191,191.0,165.0,0.0,0.0,153.899994,191,191,19,2025-11-17 13:52:00,2025-11-28 03:43:00
2,1,722851,1030,Python. Базовый уровень. Онлайн. 1 модуль,730208,1186,1,2025-10-21,0,0,602174,2025-11-07 08:23:00,2025-11-28 15:46:00,2026-05-07,<NA>,151.0,164.0,<NA>,2025-10-19 10:33:00,2025-12-26 11:20:00,8,2025-11-04 10:06:00,Europe/Moscow,0,1041.0,45.260868,121,121.0,121.0,1,181.0,2025-11-07,2026-05-07,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,<NA>,<NA>,NaN,NaN,NaN,NaN,NaT,NaT,NaN,NaN,NaN,NaN,NaN,NaT,NaT,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaT,NaT,<NA>,NaN,<NA>,<NA>,<NA>,<NA>,<NA>,NaT,NaT,<NA>,NaN,NaN,NaN,NaN,<NA>,<NA>,<NA>,<NA>,NaT,NaT
3,1,709724,1

## Итоги и основные выводы

### Базовая витрина

- Для задачи предсказания статуса правильная гранулярность — `user-course`, а не просто `user`.
- Основная витрина собирается из population-таблицы `modules/status_modules_complete.csv` и агрегатов по сырым LMS-логам.
- В ней хранятся два таргета: фактический `actual_target` и эвристический `heuristic_target` для модулей `1-3`.
- После сборки она проходит через QC: чинятся восстанавливаемые поля и автоматически удаляются только полностью пустые и константные признаки.

### Temporal-витрина

- В ноутбуке дополнительно строится отдельная длинная temporal-витрина уровня `module + user_id + course_id + cutoff_rule`.
- В ней есть четыре среза: `7_days`, `14_days`, `21_days`, `midpoint`.
- В temporal-версию включаются только признаки, которые можно честно посчитать на момент cutoff без future leakage.
- Фактический и эвристический таргеты также прокидываются в snapshot-строки как population-поля.

### Что получилось

- Итоговый базовый файл: `./artifacts/user_course_modeling_base.csv`.
- Итоговый temporal-файл: `./artifacts/user_course_modeling_snapshots_long.csv`.
- В ноутбуке дополнительно сохраняются QC-таблицы по дублям, связям, покрытию, пропускам, ремонту признаков и snapshot-срезам.


# Temporal-витрина

Ниже строится отдельная long-таблица со snapshot-срезами по ходу модуля.

Чтобы не тащить future leakage, в temporal-версию пока не включаются:

- агрегаты из `user_lessons`;
- user-level state-поля вроде `last_sign_in_at`, `sign_in_count`, `xp`;
- users-course state-поля вроде `wk_points`, `wk_max_points` и производных от них.

В snapshot-таблицу попадают только population-поля, включая `actual_target` и `heuristic_target`, статичные признаки курса и event-based агрегаты, которые можно честно посчитать на момент выбранного cutoff.


## Temporal snapshots: 7 / 14 / 21 дней и midpoint

Ниже строится отдельная long-таблица уровня `module + user_id + course_id + cutoff_rule`.

Идея простая: для одной и той же пары `user-course` мы сохраняем четыре версии признаков на разных этапах модуля.

Что здесь важно:

- `cutoff_rule` принимает значения `7_days`, `14_days`, `21_days`, `midpoint`;
- в каждой строке остаются только те события, которые произошли не позже `cutoff_at`;
- статичные признаки курса можно переносить как есть;
- population-поля `actual_target` и `heuristic_target` просто копируются в каждый snapshot;
- event-признаки нужно каждый раз пересчитывать на момент cutoff;
- признаки, которые уже знают итог модуля или итог курса, в temporal-витрину не попадают.

Ниже шаги разбиты так, чтобы было видно: откуда берутся даты модуля, как строятся четыре среза и как затем к ним присоединяются безопасные агрегаты.


### Шаг 1. Восстанавливаем даты начала и конца модуля

Сначала для каждой строки population строим временное окно модуля.

Логика старта:

- сначала берём `enrollment_date`;
- если он пустой, пробуем `users_courses.created_at`;
- если и он пустой, берём `cohort_started_at`;
- если даже этого нет, восстанавливаем старт через медиану внутри `module + course_id`, затем через медиану по модулю.

Логика конца:

- для `module 1-4` используем даты окончания из документации, а не `users_courses.access_finished_at`;
- для `module 1` в документации есть три волны завершения (`2025-11-30`, `2025-12-15`, `2025-12-31`), и пока в population нет явного wave-id, поэтому берём latest documented end: `2025-12-31`.

Ниже же считаем QC по этим anchor-датам: откуда они были взяты и какой длины получилось окно модуля.


In [8]:
def to_eod(value: pd.Series | pd.Timestamp) -> pd.Series | pd.Timestamp:
    value = pd.to_datetime(value)
    if isinstance(value, pd.Series):
        return value.dt.normalize() + pd.Timedelta(days=1) - pd.Timedelta(seconds=1)
    return value.normalize() + pd.Timedelta(days=1) - pd.Timedelta(seconds=1)


# Ключ long-таблицы и список snapshot-правил.
snapshot_keys = ["module", "user_id", "course_id", "cutoff_rule"]
snapshot_specs_df = pd.DataFrame({"cutoff_rule": ["7_days", "14_days", "21_days", "midpoint"]})

# Если enrollment_date пустой, пробуем восстановить старт через cohort + course.
cohort_start_df = (
    groups_df
    .merge(lesson_course_df[["lesson_id", "course_id"]], on="lesson_id", how="left")
    .groupby(["group_template_id", "course_id"], dropna=False, observed=True)["wk_actual_started_at"]
    .min()
    .reset_index()
    .rename(columns={"group_template_id": "cohort_id", "wk_actual_started_at": "cohort_started_at"})
)

# Собираем в одну таблицу все кандидаты на старт и конец модуля.
snapshot_population_base_df = (
    modeling_population_df
    .merge(
        users_courses_df[["user_id", "course_id", "created_at", "access_finished_at"]]
        .rename(columns={"created_at": "users_course_created_at"}),
        on=["user_id", "course_id"],
        how="left",
    )
    .merge(cohort_start_df, on=["cohort_id", "course_id"], how="left")
)

# Приоритет старта: enrollment_date -> users_courses.created_at -> cohort_started_at.
snapshot_population_base_df["module_start_at"] = snapshot_population_base_df["enrollment_date"].dt.normalize()
snapshot_population_base_df["module_start_at"] = snapshot_population_base_df["module_start_at"].fillna(
    snapshot_population_base_df["users_course_created_at"].dt.normalize()
)
snapshot_population_base_df["module_start_at"] = snapshot_population_base_df["module_start_at"].fillna(
    snapshot_population_base_df["cohort_started_at"].dt.normalize()
)

# Фоллбэки нужны только там, где прямой даты старта нет.
module_course_start_df = (
    snapshot_population_base_df.loc[snapshot_population_base_df["module_start_at"].notna()]
    .groupby(["module", "course_id"], observed=True)["module_start_at"]
    .median()
    .reset_index()
    .rename(columns={"module_start_at": "module_course_start_at"})
)
module_start_fallback_df = (
    snapshot_population_base_df.loc[snapshot_population_base_df["module_start_at"].notna()]
    .groupby("module", observed=True)["module_start_at"]
    .median()
    .reset_index()
    .rename(columns={"module_start_at": "module_start_fallback_at"})
)

snapshot_population_base_df = (
    snapshot_population_base_df
    .merge(module_course_start_df, on=["module", "course_id"], how="left")
    .merge(module_start_fallback_df, on="module", how="left")
)

# Дозаполняем старт сначала медианой внутри module + course, потом медианой по модулю.
start_missing_mask = snapshot_population_base_df["module_start_at"].isna()
snapshot_population_base_df.loc[start_missing_mask, "module_start_at"] = snapshot_population_base_df.loc[
    start_missing_mask, "module_course_start_at"
]
start_missing_mask = snapshot_population_base_df["module_start_at"].isna()
snapshot_population_base_df.loc[start_missing_mask, "module_start_at"] = snapshot_population_base_df.loc[
    start_missing_mask, "module_start_fallback_at"
]

snapshot_population_base_df["module_start_source"] = np.select(
    [
        snapshot_population_base_df["enrollment_date"].notna(),
        snapshot_population_base_df["users_course_created_at"].notna(),
        snapshot_population_base_df["cohort_started_at"].notna(),
        snapshot_population_base_df["module_course_start_at"].notna(),
        snapshot_population_base_df["module_start_fallback_at"].notna(),
    ],
    [
        "enrollment_date",
        "users_courses.created_at",
        "cohort_started_at",
        "module_course_start_median",
        "module_start_median",
    ],
    default="missing",
)

# Конец модуля берём из документации, а не из users_courses.access_finished_at.
# Для module 1 в docs есть три волны завершения: 2025-11-30, 2025-12-15 и 2025-12-31.
# Пока в population нет явного признака волны, используем latest documented end,
# чтобы midpoint и clipping не зависели от post-hoc course access-поля.
module_end_dates_from_docs = {
    1: pd.Timestamp("2025-12-31 23:59:59"),
    2: pd.Timestamp("2026-02-15 23:59:59"),
    3: pd.Timestamp("2026-03-31 23:59:59"),
    4: pd.Timestamp("2026-05-15 23:59:59"),
}
module_end_source_from_docs = {
    1: "docs.schedule.module_1_latest_wave",
    2: "docs.schedule",
    3: "docs.schedule",
    4: "docs.schedule",
}
snapshot_population_base_df["module_end_at"] = pd.to_datetime(
    snapshot_population_base_df["module"].map(module_end_dates_from_docs)
)
snapshot_population_base_df["module_end_source"] = (
    snapshot_population_base_df["module"]
    .map(module_end_source_from_docs)
    .fillna("missing")
)

# Страховка от отрицательного окна: конец не может быть раньше старта.
invalid_window_mask = snapshot_population_base_df["module_end_at"] < to_eod(snapshot_population_base_df["module_start_at"])
snapshot_population_base_df.loc[invalid_window_mask, "module_end_at"] = to_eod(
    snapshot_population_base_df.loc[invalid_window_mask, "module_start_at"]
)
snapshot_population_base_df.loc[invalid_window_mask, "module_end_source"] = (
    snapshot_population_base_df.loc[invalid_window_mask, "module_end_source"] + "_clipped_to_start"
)

snapshot_population_base_df["module_days"] = (
    snapshot_population_base_df["module_end_at"].dt.normalize()
    - snapshot_population_base_df["module_start_at"].dt.normalize()
).dt.days + 1

snapshot_anchor_quality_df = pd.DataFrame(
    [
        {"metric": "rows_total", "value": int(snapshot_population_base_df.shape[0])},
        {"metric": "module_start_at_nonnull", "value": int(snapshot_population_base_df["module_start_at"].notna().sum())},
        {"metric": "module_end_at_nonnull", "value": int(snapshot_population_base_df["module_end_at"].notna().sum())},
        {"metric": "invalid_windows_clipped", "value": int(invalid_window_mask.sum())},
        {"metric": "module_1_latest_end_at_from_docs", "value": module_end_dates_from_docs[1]},
        {"metric": "module_4_end_at_from_docs", "value": module_end_dates_from_docs[4]},
    ]
)

snapshot_anchor_sources_df = (
    snapshot_population_base_df
    .groupby(["module", "module_start_source", "module_end_source"], dropna=False, observed=True)
    .agg(
        rows=("module", "size"),
        module_days_median=("module_days", "median"),
        module_days_min=("module_days", "min"),
        module_days_max=("module_days", "max"),
    )
    .reset_index()
    .sort_values(["module", "rows"], ascending=[True, False])
    .reset_index(drop=True)
)

display(snapshot_anchor_quality_df)
display(snapshot_anchor_sources_df)


,metric,value
0,rows_total,8401
1,module_start_at_nonnull,8401
2,module_end_at_nonnull,8401
3,invalid_windows_clipped,0
4,module_1_latest_end_at_from_docs,2025-12-31 23:59:59
5,module_4_end_at_from_docs,2026-05-15 23:59:59


,module,module_start_source,module_end_source,rows,module_days_median,module_days_min,module_days_max
0,1,enrollment_date,docs.schedule.module_1_latest_wave,2972,91.0,48,104
1,2,users_courses.created_at,docs.schedule,1955,67.0,26,68
2,3,users_courses.created_at,docs.schedule,1785,44.0,43,48
3,4,cohort_started_at,docs.schedule,1374,42.0,39,43
4,4,module_course_start_median,docs.schedule,315,41.0,41,42


### Шаг 2. Строим population для четырёх cutoff-срезов

Теперь каждую строку population разворачиваем в четыре snapshot-версии:

- `7_days`
- `14_days`
- `21_days`
- `midpoint`

Для каждой версии считаем `requested_cutoff_at`, а затем берём `cutoff_at = min(requested_cutoff_at, module_end_at)`.

Это нужно, чтобы early-cutoff не выходил за границы модуля, а midpoint всегда оставался внутри реального окна.

В конце шага строим маленький инвентарь, который показывает число строк и диапазон дат по каждому `cutoff_rule`.


In [9]:
# Эти поля остаются в каждой snapshot-строке как базовая population-часть.
snapshot_population_export_cols = [
    "module",
    "user_id",
    "course_id",
    "cutoff_rule",
    "club_name",
    "teacher_id",
    "cohort_id",
    "level_bin",
    "actual_target",
    "heuristic_target",
    "module_start_at",
    "cutoff_at",
]

# Разворачиваем каждую строку population в четыре cutoff-версии.
snapshot_population_df = snapshot_population_base_df.merge(snapshot_specs_df, how="cross")

# Для fixed-day cutoff ставим конец соответствующего дня от старта модуля.
snapshot_population_df["requested_cutoff_at"] = pd.NaT
snapshot_population_df.loc[snapshot_population_df["cutoff_rule"].eq("7_days"), "requested_cutoff_at"] = to_eod(
    snapshot_population_df.loc[snapshot_population_df["cutoff_rule"].eq("7_days"), "module_start_at"] + pd.Timedelta(days=6)
)
snapshot_population_df.loc[snapshot_population_df["cutoff_rule"].eq("14_days"), "requested_cutoff_at"] = to_eod(
    snapshot_population_df.loc[snapshot_population_df["cutoff_rule"].eq("14_days"), "module_start_at"] + pd.Timedelta(days=13)
)
snapshot_population_df.loc[snapshot_population_df["cutoff_rule"].eq("21_days"), "requested_cutoff_at"] = to_eod(
    snapshot_population_df.loc[snapshot_population_df["cutoff_rule"].eq("21_days"), "module_start_at"] + pd.Timedelta(days=20)
)

# Для midpoint берём середину между module_start_at и module_end_at.
midpoint_days = (
    (
        snapshot_population_df.loc[snapshot_population_df["cutoff_rule"].eq("midpoint"), "module_end_at"].dt.normalize()
        - snapshot_population_df.loc[snapshot_population_df["cutoff_rule"].eq("midpoint"), "module_start_at"].dt.normalize()
    ).dt.days // 2
).clip(lower=0)
snapshot_population_df.loc[snapshot_population_df["cutoff_rule"].eq("midpoint"), "requested_cutoff_at"] = to_eod(
    snapshot_population_df.loc[snapshot_population_df["cutoff_rule"].eq("midpoint"), "module_start_at"]
    + pd.to_timedelta(midpoint_days, unit="D")
)

# Итоговый cutoff не должен выходить за конец модуля.
snapshot_population_df["cutoff_at"] = snapshot_population_df[["requested_cutoff_at", "module_end_at"]].min(axis=1)

# Небольшой QC: сколько строк и какой диапазон дат получился по каждому правилу.
snapshot_cutoff_inventory_df = (
    snapshot_population_df.groupby(["cutoff_rule", "module"], dropna=False, observed=True)
    .agg(
        rows=("module", "size"),
        min_cutoff_at=("cutoff_at", "min"),
        max_cutoff_at=("cutoff_at", "max"),
    )
    .reset_index()
)

display(snapshot_cutoff_inventory_df)


,cutoff_rule,module,rows,min_cutoff_at,max_cutoff_at
0,14_days,1,2972,2025-10-02 23:59:59,2025-11-27 23:59:59
1,14_days,2,1955,2025-12-23 23:59:59,2026-02-03 23:59:59
2,14_days,3,1785,2026-02-25 23:59:59,2026-03-02 23:59:59
3,14_days,4,1689,2026-04-16 23:59:59,2026-04-20 23:59:59
4,21_days,1,2972,2025-10-09 23:59:59,2025-12-04 23:59:59
5,21_days,2,1955,2025-12-30 23:59:59,2026-02-10 23:59:59
6,21_days,3,1785,2026-03-04 23:59:59,2026-03-09 23:59:59
7,21_days,4,1689,2026-04-23 23:59:59,2026-04-27 23:59:59
8,7_days,1,2972,2025-09-25 23:59:59,2025-11-20 23:59:59
9,7_days,2,1955,2025-12-16 23:59:59,2026-01-27 23:59:59


### Шаг 3. Подготавливаем безопасные источники

На этом шаге мы ещё не считаем финальные признаки. Мы только готовим источники, из которых их потом можно честно собрать на момент cutoff.

Что здесь происходит:

- дочиняются статичные course-level признаки;
- сырые event-таблицы сужаются до ключей population;
- из каждого источника оставляются только нужные колонки;
- выкидываются признаки, которые знают будущее и поэтому опасны для temporal-срезов.

Важно: здесь мы сознательно не тащим в snapshot-витрину `user_lessons`, user-level state-поля и users-course state-поля, потому что в текущем виде они отражают итоговое состояние, а не состояние на конкретный день модуля.


In [10]:
# Course-level признаки статичны, поэтому их можно подготовить один раз.
snapshot_course_features_df = course_features_df.merge(course_duration_proxy_df, on="course_id", how="left")

snapshot_total_replace_mask = (
    snapshot_course_features_df["course_video_duration_total"].fillna(0).eq(0)
    & snapshot_course_features_df["course_video_duration_total_fixed"].notna()
)
snapshot_mean_replace_mask = (
    snapshot_course_features_df["course_video_duration_mean"].isna()
    & snapshot_course_features_df["course_video_duration_mean_fixed"].notna()
)

snapshot_course_features_df.loc[snapshot_total_replace_mask, "course_video_duration_total"] = snapshot_course_features_df.loc[
    snapshot_total_replace_mask, "course_video_duration_total_fixed"
]
snapshot_course_features_df.loc[snapshot_mean_replace_mask, "course_video_duration_mean"] = snapshot_course_features_df.loc[
    snapshot_mean_replace_mask, "course_video_duration_mean_fixed"
]
snapshot_course_features_df = snapshot_course_features_df.drop(
    columns=["course_video_duration_total_fixed", "course_video_duration_mean_fixed"],
    errors="ignore",
)

# В actions приводим user_id к одному имени, чтобы дальше все merge были одинаковыми.
snapshot_actions_course_df = actions_course_df.drop(columns=["user_id"], errors="ignore").rename(
    columns={"user_id_from_uc": "user_id"}
)

# Дальше режем каждый сырой источник до population-ключей и минимального набора колонок.
snapshot_population_keys_df = modeling_population_df[["user_id", "course_id"]].drop_duplicates()

snapshot_access_course_df = access_course_df[[
    "users_course_id",
    "user_id",
    "course_id",
    "access_started_at",
    "access_expired_at",
]].merge(snapshot_population_keys_df, on=["user_id", "course_id"], how="inner")

snapshot_user_activity_course_df = user_activity_course_df.loc[
    user_activity_course_df["course_id"].notna(),
    ["user_lesson_id", "user_id", "course_id", "action", "created_at"],
].merge(snapshot_population_keys_df, on=["user_id", "course_id"], how="inner")

snapshot_actions_course_df = snapshot_actions_course_df.loc[
    snapshot_actions_course_df["course_id"].notna(),
    ["user_id", "course_id", "action", "created_at"],
].merge(snapshot_population_keys_df, on=["user_id", "course_id"], how="inner")

snapshot_media_course_df = media_course_df.loc[
    media_course_df["course_id"].notna(),
    ["user_id", "course_id", "resource_id", "watch_share", "started_at"],
].merge(snapshot_population_keys_df, on=["user_id", "course_id"], how="inner")

snapshot_user_trainings_course_df = user_trainings_course_df.loc[
    user_trainings_course_df["course_id"].notna(),
    [
        "user_id",
        "course_id",
        "training_id",
        "solved_tasks_count",
        "earned_points",
        "submitted_answers_count",
        "started_at",
        "finished_at",
        "attempts",
        "mark",
    ],
].merge(snapshot_population_keys_df, on=["user_id", "course_id"], how="inner")

snapshot_user_answers_course_df = user_answers_course_df.loc[
    user_answers_course_df["course_id"].notna(),
    [
        "user_id",
        "course_id",
        "task_id",
        "attempts",
        "solved",
        "points",
        "max_attempts",
        "skipped",
        "submitted_at",
        "wk_partial_answer",
        "async_check_status",
    ],
].merge(snapshot_population_keys_df, on=["user_id", "course_id"], how="inner")


### Шаг 4. Описываем агрегации на момент cutoff

Дальше идут функции-агрегаторы. Каждая из них делает одну и ту же базовую операцию:

1. берёт один источник;
2. матчится к snapshot-population;
3. оставляет только события не позже `cutoff_at`;
4. считает агрегаты на уровне `module + user_id + course_id + cutoff_rule`.

Ниже функции разбиты по источникам:

- `aggregate_access_snapshot` — доступы;
- `aggregate_user_activity_snapshot` — lesson activity;
- `aggregate_actions_snapshot` — course actions;
- `aggregate_media_snapshot` — просмотры media;
- `aggregate_answers_snapshot` — ответы на задания;
- `aggregate_trainings_snapshot` — старты и завершения trainings.

Так temporal-логика остаётся прозрачной: каждый блок признаков собирается отдельно и не смешивает события из будущего.


In [11]:
# Удобный шаблон: если по источнику нет событий до cutoff, возвращаем пустой DataFrame с правильной схемой.
def empty_snapshot_df(value_columns: list[str]) -> pd.DataFrame:
    return pd.DataFrame(columns=snapshot_keys + value_columns)


# Access-агрегаты считаются не по полному окну доступа, а по наблюдаемой части до cutoff.
def aggregate_access_snapshot(snapshot_pop_slice_df: pd.DataFrame) -> pd.DataFrame:
    joined_df = snapshot_access_course_df.merge(
        snapshot_pop_slice_df[snapshot_keys + ["cutoff_at"]],
        on=["user_id", "course_id"],
        how="inner",
    )
    joined_df = joined_df.loc[
        joined_df["access_started_at"].notna()
        & joined_df["access_started_at"].le(joined_df["cutoff_at"])
    ].copy()
    if joined_df.empty:
        return empty_snapshot_df(
            ["access_periods_count", "access_days_observed_total", "access_started_at_first", "access_observed_at_last"]
        )

    # Если доступ длится дольше cutoff или дата конца пустая, обрезаем период по cutoff.
    joined_df["observed_access_end_at"] = joined_df["access_expired_at"]
    end_fill_mask = (
        joined_df["observed_access_end_at"].isna()
        | joined_df["observed_access_end_at"].gt(joined_df["cutoff_at"])
    )
    joined_df.loc[end_fill_mask, "observed_access_end_at"] = joined_df.loc[end_fill_mask, "cutoff_at"]
    joined_df["observed_access_days"] = (
        joined_df["observed_access_end_at"].dt.normalize()
        - joined_df["access_started_at"].dt.normalize()
    ).dt.days.clip(lower=0)

    return (
        joined_df.groupby(snapshot_keys, observed=True)
        .agg(
            access_periods_count=("users_course_id", "size"),
            access_days_observed_total=("observed_access_days", "sum"),
            access_started_at_first=("access_started_at", "min"),
            access_observed_at_last=("observed_access_end_at", "max"),
        )
        .reset_index()
    )


# Lesson activity: считаем общий объём событий, активные дни и раскладку по action.
def aggregate_user_activity_snapshot(snapshot_pop_slice_df: pd.DataFrame) -> pd.DataFrame:
    joined_df = snapshot_user_activity_course_df.merge(
        snapshot_pop_slice_df[snapshot_keys + ["cutoff_at"]],
        on=["user_id", "course_id"],
        how="inner",
    )
    joined_df = joined_df.loc[
        joined_df["created_at"].notna()
        & joined_df["created_at"].le(joined_df["cutoff_at"])
    ].copy()
    if joined_df.empty:
        return empty_snapshot_df(
            [
                "lesson_activity_events_total",
                "lesson_activity_user_lessons_nunique",
                "lesson_activity_days",
                "lesson_activity_first_at",
                "lesson_activity_last_at",
            ]
        )

    agg_df = (
        joined_df.groupby(snapshot_keys, observed=True)
        .agg(
            lesson_activity_events_total=("action", "size"),
            lesson_activity_user_lessons_nunique=("user_lesson_id", "nunique"),
            lesson_activity_days=("created_at", lambda s: s.dt.normalize().nunique()),
            lesson_activity_first_at=("created_at", "min"),
            lesson_activity_last_at=("created_at", "max"),
        )
        .reset_index()
    )
    # Отдельно разворачиваем counts по каждому action в wide-формат.
    counts_wide_df = (
        joined_df[snapshot_keys + ["action"]]
        .assign(event_count=1)
        .pivot_table(
            index=snapshot_keys,
            columns="action",
            values="event_count",
            aggfunc="sum",
            fill_value=0,
        )
        .rename_axis(columns=None)
        .add_prefix("lesson_activity_count__")
        .reset_index()
    )
    return agg_df.merge(counts_wide_df, on=snapshot_keys, how="left")


# Course actions: та же схема, но уже на логе действий по user-course.
def aggregate_actions_snapshot(snapshot_pop_slice_df: pd.DataFrame) -> pd.DataFrame:
    joined_df = snapshot_actions_course_df.merge(
        snapshot_pop_slice_df[snapshot_keys + ["cutoff_at"]],
        on=["user_id", "course_id"],
        how="inner",
    )
    joined_df = joined_df.loc[
        joined_df["created_at"].notna()
        & joined_df["created_at"].le(joined_df["cutoff_at"])
    ].copy()
    if joined_df.empty:
        return empty_snapshot_df(
            ["actions_total", "action_days", "action_first_at", "action_last_at"]
        )

    agg_df = (
        joined_df.groupby(snapshot_keys, observed=True)
        .agg(
            actions_total=("action", "size"),
            action_days=("created_at", lambda s: s.dt.normalize().nunique()),
            action_first_at=("created_at", "min"),
            action_last_at=("created_at", "max"),
        )
        .reset_index()
    )
    counts_wide_df = (
        joined_df[snapshot_keys + ["action"]]
        .assign(event_count=1)
        .pivot_table(
            index=snapshot_keys,
            columns="action",
            values="event_count",
            aggfunc="sum",
            fill_value=0,
        )
        .rename_axis(columns=None)
        .add_prefix("action_count__")
        .reset_index()
    )
    return agg_df.merge(counts_wide_df, on=snapshot_keys, how="left")


In [12]:
# Media: считаем количество сессий, разнообразие ресурсов и глубину просмотра.
def aggregate_media_snapshot(snapshot_pop_slice_df: pd.DataFrame) -> pd.DataFrame:
    joined_df = snapshot_media_course_df.merge(
        snapshot_pop_slice_df[snapshot_keys + ["cutoff_at"]],
        on=["user_id", "course_id"],
        how="inner",
    )
    joined_df = joined_df.loc[
        joined_df["started_at"].notna()
        & joined_df["started_at"].le(joined_df["cutoff_at"])
    ].copy()
    if joined_df.empty:
        return empty_snapshot_df(
            [
                "media_sessions_total",
                "media_resources_nunique",
                "media_watch_share_mean",
                "media_watch_share_max",
                "media_sessions_80pct",
                "media_sessions_full",
                "media_started_at_first",
                "media_started_at_last",
            ]
        )

    return (
        joined_df.groupby(snapshot_keys, observed=True)
        .agg(
            media_sessions_total=("resource_id", "size"),
            media_resources_nunique=("resource_id", "nunique"),
            media_watch_share_mean=("watch_share", "mean"),
            media_watch_share_max=("watch_share", "max"),
            media_sessions_80pct=("watch_share", lambda s: int((s >= 0.8).sum())),
            media_sessions_full=("watch_share", lambda s: int((s >= 1.0).sum())),
            media_started_at_first=("started_at", "min"),
            media_started_at_last=("started_at", "max"),
        )
        .reset_index()
    )


# Answers: фиксируем объём попыток, solved/skipped/partial и суммарные очки на момент cutoff.
def aggregate_answers_snapshot(snapshot_pop_slice_df: pd.DataFrame) -> pd.DataFrame:
    joined_df = snapshot_user_answers_course_df.merge(
        snapshot_pop_slice_df[snapshot_keys + ["cutoff_at"]],
        on=["user_id", "course_id"],
        how="inner",
    )
    joined_df = joined_df.loc[
        joined_df["submitted_at"].notna()
        & joined_df["submitted_at"].le(joined_df["cutoff_at"])
    ].copy()
    if joined_df.empty:
        return empty_snapshot_df(
            [
                "answer_rows",
                "answer_tasks_nunique",
                "answer_solved_count",
                "answer_skipped_count",
                "answer_partial_count",
                "answer_points_sum",
                "answer_attempts_sum",
                "answer_max_attempts_sum",
                "answer_async_nonzero_count",
                "answer_first_submitted_at",
                "answer_last_submitted_at",
            ]
        )

    return (
        joined_df.groupby(snapshot_keys, observed=True)
        .agg(
            answer_rows=("task_id", "size"),
            answer_tasks_nunique=("task_id", "nunique"),
            answer_solved_count=("solved", bool_sum),
            answer_skipped_count=("skipped", bool_sum),
            answer_partial_count=("wk_partial_answer", bool_sum),
            answer_points_sum=("points", "sum"),
            answer_attempts_sum=("attempts", "sum"),
            answer_max_attempts_sum=("max_attempts", "sum"),
            answer_async_nonzero_count=("async_check_status", lambda s: int((s.fillna(0) != 0).sum())),
            answer_first_submitted_at=("submitted_at", "min"),
            answer_last_submitted_at=("submitted_at", "max"),
        )
        .reset_index()
    )


# Trainings разделяем на два потока: что уже стартовало и что уже успело завершиться к cutoff.
def aggregate_trainings_snapshot(snapshot_pop_slice_df: pd.DataFrame) -> pd.DataFrame:
    joined_df = snapshot_user_trainings_course_df.merge(
        snapshot_pop_slice_df[snapshot_keys + ["cutoff_at"]],
        on=["user_id", "course_id"],
        how="inner",
    )
    started_df = joined_df.loc[
        joined_df["started_at"].notna()
        & joined_df["started_at"].le(joined_df["cutoff_at"])
    ].copy()
    finished_df = joined_df.loc[
        joined_df["finished_at"].notna()
        & joined_df["finished_at"].le(joined_df["cutoff_at"])
    ].copy()

    # Стартовавшие trainings дают ранний сигнал вовлечённости, даже если они ещё не завершены.
    started_agg_df = empty_snapshot_df(
        ["trainings_started_total", "trainings_started_nunique", "training_started_at_first"]
    )
    if not started_df.empty:
        started_agg_df = (
            started_df.groupby(snapshot_keys, observed=True)
            .agg(
                trainings_started_total=("training_id", "size"),
                trainings_started_nunique=("training_id", "nunique"),
                training_started_at_first=("started_at", "min"),
            )
            .reset_index()
        )

    # Outcome-метрики считаем только по тем trainings, которые завершились не позже cutoff.
    finished_agg_df = empty_snapshot_df(
        [
            "trainings_finished_total",
            "trainings_finished_nunique",
            "training_solved_tasks_total",
            "training_earned_points_sum",
            "training_submitted_answers_sum",
            "training_attempts_sum",
            "training_mark_max",
            "training_finished_at_last",
        ]
    )
    if not finished_df.empty:
        finished_agg_df = (
            finished_df.groupby(snapshot_keys, observed=True)
            .agg(
                trainings_finished_total=("training_id", "size"),
                trainings_finished_nunique=("training_id", "nunique"),
                training_solved_tasks_total=("solved_tasks_count", "sum"),
                training_earned_points_sum=("earned_points", "sum"),
                training_submitted_answers_sum=("submitted_answers_count", "sum"),
                training_attempts_sum=("attempts", "sum"),
                training_mark_max=("mark", "max"),
                training_finished_at_last=("finished_at", "max"),
            )
            .reset_index()
        )

    return started_agg_df.merge(finished_agg_df, on=snapshot_keys, how="outer")


### Шаг 5. Собираем long-таблицу, QC и экспорт

В финале:

- считаем snapshot-витрину отдельно для каждого `cutoff_rule`;
- склеиваем их в одну long-таблицу;
- считаем QC по размеру, покрытию и пропускам;
- для проверок заполненности исключаем `module == 4`, потому что по нему в LMS-источниках структурно нет event-данных;
- сохраняем итоговый файл `./artifacts/user_course_modeling_snapshots_long.csv`.

Что показывают итоговые таблицы:

- `snapshot_summary_df` — размер long-таблицы и диапазон cutoff-дат;
- в саму snapshot-витрину, кроме признаков, попадают и population-поля `actual_target` / `heuristic_target`;
- `snapshot_coverage_df` — покрытие основных блоков признаков;
- `snapshot_missingness_preview_df` — самые проблемные поля по `na_share` внутри каждого cutoff;
- `snapshot_feature_policy_df` — какие блоки включены, а какие сознательно исключены и почему.


In [13]:
# Для каждого cutoff_rule собираем отдельный snapshot-слой, а потом склеиваем их в одну long-таблицу.
def build_snapshot_dataset_for_rule(cutoff_rule: str) -> pd.DataFrame:
    snapshot_pop_slice_df = snapshot_population_df.loc[
        snapshot_population_df["cutoff_rule"].eq(cutoff_rule)
    ].copy()

    # Сначала берём population-часть, затем по очереди присоединяем статичные и event-блоки.
    return (
        snapshot_pop_slice_df[snapshot_population_export_cols]
        .merge(snapshot_course_features_df, on="course_id", how="left")
        .merge(aggregate_access_snapshot(snapshot_pop_slice_df), on=snapshot_keys, how="left")
        .merge(aggregate_user_activity_snapshot(snapshot_pop_slice_df), on=snapshot_keys, how="left")
        .merge(aggregate_actions_snapshot(snapshot_pop_slice_df), on=snapshot_keys, how="left")
        .merge(aggregate_media_snapshot(snapshot_pop_slice_df), on=snapshot_keys, how="left")
        .merge(aggregate_trainings_snapshot(snapshot_pop_slice_df), on=snapshot_keys, how="left")
        .merge(aggregate_answers_snapshot(snapshot_pop_slice_df), on=snapshot_keys, how="left")
    )


# Собираем четыре среза и склеиваем их в один long DataFrame.
snapshot_frames = [build_snapshot_dataset_for_rule(cutoff_rule) for cutoff_rule in snapshot_specs_df["cutoff_rule"]]
snapshot_modeling_dataset_df = pd.concat(snapshot_frames, ignore_index=True, sort=False)
snapshot_modeling_dataset_df = snapshot_modeling_dataset_df.sort_values(
    ["module", "user_id", "course_id", "cutoff_rule"]
).reset_index(drop=True)

# Summary показывает размер итоговой таблицы и диапазон cutoff-дат по каждому правилу.
snapshot_summary_df = (
    snapshot_modeling_dataset_df.groupby("cutoff_rule", dropna=False, observed=True)
    .agg(
        rows=("cutoff_rule", "size"),
        modules=("module", "nunique"),
        labeled_rows=("actual_target", lambda s: int(s.notna().sum())),
        min_cutoff_at=("cutoff_at", "min"),
        max_cutoff_at=("cutoff_at", "max"),
    )
    .reset_index()
)
snapshot_summary_df["columns"] = int(snapshot_modeling_dataset_df.shape[1])

# Для coverage/missingness исключаем module 4: по нему в LMS-источниках структурно нет event-данных.
snapshot_quality_base_df = snapshot_modeling_dataset_df.loc[
    snapshot_modeling_dataset_df["module"] != 4
].copy()

# Coverage показывает, у какой доли строк есть хотя бы какое-то заполнение по каждому блоку признаков.
snapshot_coverage_df = (
    snapshot_quality_base_df.groupby("cutoff_rule", dropna=False, observed=True)
    .agg(
        rows=("cutoff_rule", "size"),
        course_feature_rows=("course_video_duration_total", lambda s: int(s.notna().sum())),
        access_feature_rows=("access_periods_count", lambda s: int(s.notna().sum())),
        user_activity_feature_rows=("lesson_activity_events_total", lambda s: int(s.notna().sum())),
        actions_feature_rows=("actions_total", lambda s: int(s.notna().sum())),
        media_feature_rows=("media_sessions_total", lambda s: int(s.notna().sum())),
        trainings_feature_rows=("trainings_started_total", lambda s: int(s.notna().sum())),
        answers_feature_rows=("answer_rows", lambda s: int(s.notna().sum())),
    )
    .reset_index()
)
for col in [
    "course_feature_rows",
    "access_feature_rows",
    "user_activity_feature_rows",
    "actions_feature_rows",
    "media_feature_rows",
    "trainings_feature_rows",
    "answers_feature_rows",
]:
    snapshot_coverage_df[col.replace("_rows", "_pct")] = snapshot_coverage_df.apply(
        lambda row, src=col: pct(int(row[src]), int(row["rows"])),
        axis=1,
    )

# Missingness считаем уже на базе без module 4 и оставляем только na_share.
snapshot_missingness_frames = []
for cutoff_rule, frame in snapshot_quality_base_df.groupby("cutoff_rule", dropna=False, observed=True):
    missingness_df = (
        frame.isna()
        .mean()
        .sort_values(ascending=False)
        .rename("na_share")
        .reset_index()
        .rename(columns={"index": "column"})
    )
    missingness_df["cutoff_rule"] = cutoff_rule
    snapshot_missingness_frames.append(missingness_df)
snapshot_missingness_df = pd.concat(snapshot_missingness_frames, ignore_index=True)
snapshot_missingness_preview_df = (
    snapshot_missingness_df.sort_values(["cutoff_rule", "na_share"], ascending=[True, False])
    .groupby("cutoff_rule", group_keys=False)
    .head(15)
    .reset_index(drop=True)
)

# Отдельно фиксируем, какие блоки попали в temporal-витрину, а какие сознательно исключены.
snapshot_feature_policy_df = pd.DataFrame(
    [
        {"feature_block": "course_features_df + duration proxy", "status": "included", "reason": "Статичные признаки курса не зависят от будущих событий пользователя."},
        {"feature_block": "access_course_df", "status": "included", "reason": "Используются только периоды, начавшиеся не позже cutoff; длительность обрезается по cutoff."},
        {"feature_block": "user_activity_histories_df", "status": "included", "reason": "События фильтруются по created_at <= cutoff_at."},
        {"feature_block": "wk_users_courses_actions_df", "status": "included", "reason": "События фильтруются по created_at <= cutoff_at."},
        {"feature_block": "wk_media_view_sessions_df", "status": "included", "reason": "События фильтруются по started_at <= cutoff_at."},
        {"feature_block": "user_trainings_df", "status": "included", "reason": "Outcome-признаки считаются только для finished_at <= cutoff_at; отдельно сохраняется число стартовавших тренировок."},
        {"feature_block": "user_answers_df", "status": "included", "reason": "События фильтруются по submitted_at <= cutoff_at."},
        {"feature_block": "user_lessons_agg_df", "status": "excluded", "reason": "В текущем виде это итоговое состояние уроков без честной temporal-логики на выбранный cutoff."},
        {"feature_block": "user_profile_df", "status": "excluded", "reason": "Поля вроде last_sign_in_at и sign_in_count знают будущее относительно ранних snapshot-срезов."},
        {"feature_block": "users_courses_feature_df", "status": "excluded", "reason": "Поля вроде wk_points и производных отражают итоговое состояние курса, а не состояние на cutoff."},
    ]
)

# Сохраняем готовую long-таблицу и выводим ключевые QC-таблицы для чтения в ноутбуке.
snapshot_export_path = OUTPUT_DIR / "user_course_modeling_snapshots_long.csv"
snapshot_modeling_dataset_df.to_csv(snapshot_export_path, index=False)

display(snapshot_summary_df)
display(snapshot_coverage_df)
display(snapshot_missingness_preview_df)
display(snapshot_feature_policy_df)
display(snapshot_modeling_dataset_df.head())


,cutoff_rule,rows,modules,labeled_rows,min_cutoff_at,max_cutoff_at,columns
0,14_days,8401,4,4927,2025-10-02 23:59:59,2026-04-20 23:59:59,69
1,21_days,8401,4,4927,2025-10-09 23:59:59,2026-04-27 23:59:59,69
2,7_days,8401,4,4927,2025-09-25 23:59:59,2026-04-13 23:59:59,69
3,midpoint,8401,4,4927,2025-11-09 23:59:59,2026-04-26 23:59:59,69


,cutoff_rule,rows,course_feature_rows,access_feature_rows,user_activity_feature_rows,actions_feature_rows,media_feature_rows,trainings_feature_rows,answers_feature_rows,course_feature_pct,access_feature_pct,user_activity_feature_pct,actions_feature_pct,media_feature_pct,trainings_feature_pct,answers_feature_pct
0,14_days,6712,6712.0,4757,2991,2993,2868,23,2334,100.0,70.87,44.56,44.59,42.73,0.34,34.77
1,21_days,6712,6712.0,5050,3551,3563,3455,262,2759,100.0,75.24,52.91,53.08,51.47,3.90,41.11
2,7_days,6712,6712.0,4243,1942,1947,1883,2,1059,100.0,63.22,28.93,29.01,28.05,0.03,15.78
3,midpoint,6712,6712.0,6690,4956,4978,4804,811,3989,100.0,99.67,73.84,74.17,71.57,12.08,59.43


,column,na_share,cutoff_rule
0,trainings_finished_total,0.996573,14_days
1,training_finished_at_last,0.996573,14_days
2,trainings_started_total,0.996573,14_days
3,trainings_started_nunique,0.996573,14_days
4,training_started_at_first,0.996573,14_days
5,trainings_finished_nunique,0.996573,14_days
6,training_solved_tasks_total,0.996573,14_days
7,training_earned_points_sum,0.996573,14_days
8,training_submitted_answers_sum,0.996573,14_days
9,training_attempts_sum,0.996573,14_days


,feature_block,status,reason
0,course_features_df + duration proxy,included,Статичные признаки курса не зависят от будущих...
1,access_course_df,included,"Используются только периоды, начавшиеся не поз..."
2,user_activity_histories_df,included,События фильтруются по created_at <= cutoff_at.
3,wk_users_courses_actions_df,included,События фильтруются по created_at <= cutoff_at.
4,wk_media_view_sessions_df,included,События фильтруются по started_at <= cutoff_at.
5,user_trainings_df,included,Outcome-признаки считаются только для finished...
6,user_answers_df,included,События фильтруются по submitted_at <= cutoff_at.
7,user_lessons_agg_df,excluded,В текущем виде это итоговое состояние уроков б...
8,user_profile_df,excluded,Поля вроде last_sign_in_at и sign_in_count зна...
9,users_courses_feature_df,excluded,Поля вроде wk_points и производных отражают ит...


,module,user_id,course_id,cutoff_rule,club_name,teacher_id,cohort_id,level_bin,actual_target,heuristic_target,module_start_at,cutoff_at,course_video_duration_total,course_video_duration_mean,course_lesson_tasks_total,course_lesson_tasks_unique,course_required_tasks_total,access_periods_count,access_days_observed_total,access_started_at_first,access_observed_at_last,lesson_activity_events_total,lesson_activity_user_lessons_nunique,lesson_activity_days,lesson_activity_first_at,lesson_activity_last_at,lesson_activity_count__show_conspect,lesson_activity_count__visit_translation,lesson_activity_count__visit_video,actions_total,action_days,action_first_at,action_last_at,action_count__scratch_playground_visited,action_count__start_training,action_count__user_answer,action_count__visit_preparation_material,action_count__visit_translation,action_count__visit_video,media_sessions_total,media_resources_nunique,media_watch_share_mean,media_watch_share_max,media_sessions_80pct,media_sessions_full,media_started_at_first,media_started_at_last,trainings_started_total,trainings_started_nunique,training_started_at_first,trainings_finished_total,trainings_finished_nunique,training_solved_tasks_total,training_earned_points_sum,training_submitted_answers_sum,training_attempts_sum,training_mark_max,training_finished_at_last,answer_rows,answer_tasks_nunique,answer_solved_count,answer_skipped_count,answer_partial_count,answer_points_sum,answer_attempts_sum,answer_max_attempts_sum,answer_async_nonzero_count,answer_first_submitted_at,answer_last_submitted_at
0,1,700083,1029,14_days,Python. Начальный уровень. Онлайн. 1 модуль,732430,1190,0,0,0,2025-09-19,2025-10-02 23:59:59,1037.0,45.086956,110,110.0,110.0,<NA>,NaN,NaT,NaT,NaN,NaN,NaN,NaT,NaT,NaN,NaN,NaN,NaN,NaN,NaT,NaT,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaT,NaT,<NA>,NaN,NaT,<NA>,NaN,<NA>,<NA>,<NA>,<NA>,<NA>,NaT,<NA>,NaN,NaN,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,NaT,NaT
1,1,700083,1029,21_days,Python. Начальный уровень. Онлайн. 1 модуль,732430,1190,0,0,0,2025-09-19,2025-10-09 23:59:59,1037.0,45.086956,110,110.0,110.0,<NA>,NaN,NaT,NaT,NaN,NaN,NaN,NaT,NaT,NaN,NaN,NaN,NaN,NaN,NaT,NaT,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaT,NaT,<NA>,NaN,NaT,<NA>,NaN,<NA>,<NA>,<NA>,<NA>,<NA>,NaT,<NA>,NaN,NaN,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,NaT,NaT
2,1,700083,1029,7_days,Python. Начальный уровень. Онлайн. 1 модуль,732430,1190,0,0,0,2025-09-19,2025-09-25 23:59:59,1037.0,45.086956,110,110.0,110.0,<NA>,NaN,NaT,NaT,NaN,NaN,NaN,NaT,NaT,NaN,NaN,NaN,NaN,NaN,NaT,NaT,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaT,NaT,<NA>,NaN,NaT,<NA>,NaN,<NA>,<NA>,<NA>,<NA>,<NA>,NaT,<NA>,NaN,NaN,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,NaT,NaT
3,1,700083,1029,midpoint,Python. Начальный уровень. Онлайн. 1 модуль,732430,1190,0,0,0,2025-09-19,2025-11-09 23:59:59,1037.0,45.086956,110,110.0,110.0,1,2.0,2025-11-07,2025-11-09 23:59:59,NaN,NaN,NaN,NaT,NaT,NaN,NaN,NaN,NaN,NaN,NaT,NaT,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaT,NaT,<NA>,NaN,NaT,<NA>,NaN,<NA>,<NA>,<NA>,<NA>,<NA>,NaT,<NA>,NaN,NaN,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,NaT,NaT
4,1,700099,1029,14_days,Python. Начальный уровень. Онлайн. 1 модуль,699986,1191,0,0,0,2025-09-19,2025-10-02 23:59:59,1037.0,45.086956,110,110.0,110.0,<NA>,NaN,NaT,NaT,NaN,NaN,NaN,NaT,NaT,NaN,NaN,NaN,NaN,NaN,NaT,NaT,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaT,NaT,<NA>,NaN,NaT,<NA>,NaN,<NA>,<NA>,<NA>,<NA>,<NA>,NaT,<NA>,NaN,NaN,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,NaT,NaT


## Спецификация итоговых таблиц

Ниже собраны две самодостаточные спецификации итоговых CSV. Их цель — дать проверяющему, моделлеру и feature engineer понятное описание таблиц без необходимости читать код ноутбука.

Как читать эти спецификации:

- `ID` и `Label` поля обычно не идут напрямую в feature matrix;
- `Population-атрибуты` описывают контекст строки;
- `Static course feature` одинаковы для всех пользователей одного курса;
- `Aggregated ... feature` посчитаны из сырых логов LMS;
- для low-cardinality категорий значения перечислены явно, для high-cardinality указана кардинальность и смысл.


### Спецификация 1. Базовая витрина `user_course_modeling_base.csv`

| Параметр | Значение |
|---|---|
| Файл | `./artifacts/user_course_modeling_base.csv` |
| Строк | `8401` |
| Столбцов | `91` |
| Гранулярность | 1 строка = 1 объект `module + user_id + course_id`. |
| Ключ | `module + user_id + course_id` |

Ключевые замечания:

- Это полная user-course витрина без temporal-отсечения: в event- и state-полях используется вся доступная история на момент выгрузки.
- Все count-, sum- и ratio-поля в CSV могут выглядеть как `float`, потому что после left join пропуски кодируются через `NaN`.
- `actual_target` — основной фактический label; `heuristic_target` — отдельная proxy-разметка. Оба поля служебные и не должны идти в обучение как feature.
- Поля `users_course_id`, `teacher_id`, `cohort_id`, `club_name`, `timezone` и другие идентификаторы/категории полезны как grouping/context, но требуют отдельного решения перед обучением модели.

| Атрибут | Тип / роль | Смысл | Категории / важные примечания |
|---|---|---|---|
| `module` | Категория / часть ключа | Номер модуля в общей population. | Категории: {1, 2, 3, 4}. |
| `user_id` | ID / часть ключа | Идентификатор пользователя LMS. | Первичный user-level идентификатор. |
| `course_id` | ID / часть ключа | Идентификатор курса, к которому привязана строка. | Курс внутри LMS. |
| `club_name` | Категориальный population-атрибут | Текстовое название потока/клуба/группы из population-таблицы. | Высокая кардинальность: 141 значений. |
| `teacher_id` | ID / population-атрибут | Идентификатор преподавателя в population. | Можно использовать как cohort-like grouping id. |
| `cohort_id` | ID / population-атрибут | Идентификатор набора/кохорты внутри population. | Связан с group_template_id. |
| `level_bin` | Категориальный population-атрибут | Бинарный уровень программы из population. | Категории: {0, 1}. |
| `enrollment_date` | Дата / population-атрибут | Дата старта пользователя в модуле из status-таблицы. | Базовая дата начала модуля, если она известна. |
| `actual_target` | Label | Фактический бинарный таргет по известному статусу модуля. | Категории: {0.0, 1.0}; 1 = завершил, 0 = отчислен, NA = фактический статус неизвестен. Не использовать как feature. |
| `heuristic_target` | Proxy label | Эвристический бинарный таргет из `status_by_best_heuristic`. | Категории: {0.0, 1.0}; 1 = эвристически завершил, 0 = эвристически отчислен, NA = эвристики нет. Не использовать как feature. |
| `users_course_id` | ID / bridge | Идентификатор связи пользователя и курса в таблице `users_courses`. | Для `module 4` часто отсутствует. |
| `created_at` | Datetime / users_courses | Время создания записи `users_courses`. | По сути техническая дата старта course-enrollment. |
| `updated_at` | Datetime / users_courses | Время последнего обновления записи `users_courses`. | Stateful поле; может знать будущее. |
| `access_finished_at` | Дата / users_courses | Дата окончания доступа по `users_courses`. | Stateful поле; для `module 4` отсутствует. |
| `wk_points` | Numeric feature / users_courses | Текущие накопленные баллы пользователя в курсе. | Stateful поле полной витрины. |
| `wk_max_points` | Numeric feature / users_courses | Максимально возможные баллы по курсу. | Нормирующая величина для course points. |
| `wk_max_task_count` | Numeric feature / users_courses | Максимальное число задач по курсу. | Структурная course-progress величина. |
| `course_points_ratio` | Numeric feature / users_courses | Отношение `wk_points` к `wk_max_points`. | Нормализованный прогресс по баллам. |
| `user_created_at` | Datetime / user profile | Время создания пользовательского аккаунта. | Глобальный user-level атрибут. |
| `user_updated_at` | Datetime / user profile | Время последнего обновления пользовательского профиля. | Stateful поле полной витрины. |
| `sign_in_count` | Numeric feature / user profile | Число входов пользователя в LMS. | Глобальная user-level активность. |
| `user_last_sign_in_at` | Datetime / user profile | Время последнего входа пользователя в LMS. | Stateful поле полной витрины. |
| `timezone` | Категориальный user-атрибут | Часовой пояс пользователя из профиля. | Высокая кардинальность: 51 IANA/UTC значений. |
| `xp` | Numeric feature / user profile | Глобальные пользовательские XP в LMS. | Stateful поле полной витрины. |
| `course_video_duration_total` | Static course feature | Суммарная длительность видео по урокам курса. | При необходимости восстановлена через proxy из groups.duration. |
| `course_video_duration_mean` | Static course feature | Средняя длительность видео-урока в курсе. | При необходимости восстановлена через proxy из groups.duration. |
| `course_lesson_tasks_total` | Static course feature | Общее число lesson-task строк в курсе. | Структура курса, одинакова для всех пользователей курса. |
| `course_lesson_tasks_unique` | Static course feature | Число уникальных task_id в lesson_tasks курса. | Структура курса. |
| `course_required_tasks_total` | Static course feature | Число обязательных lesson-task в курсе. | Структура курса. |
| `access_periods_count` | Aggregated access feature | Число периодов доступа, привязанных к user-course. | Полная история по всем найденным access-периодам. |
| `access_days_total` | Aggregated access feature | Суммарная длительность всех access-периодов в днях. | Полная длительность доступа в базовой витрине. |
| `access_started_at_first` | Aggregated access feature | Самая ранняя дата начала доступа. | Минимум по всем access-периодам. |
| `access_expired_at_last` | Aggregated access feature | Самая поздняя дата окончания доступа. | Максимум по всем access-периодам. |
| `user_lessons_rows` | Aggregated lesson feature | Число строк `user_lessons`, связанных с user-course. | Есть только в полной витрине; в snapshot сознательно исключено. |
| `user_lessons_nunique` | Aggregated lesson feature | Число уникальных `user_lesson_id` в курсе. | Есть только в полной витрине. |
| `lessons_logged_nunique` | Aggregated lesson feature | Число уникальных уроков, где у пользователя есть `user_lessons`-лог. | Есть только в полной витрине. |
| `groups_logged_nunique` | Aggregated lesson feature | Число уникальных групп в `user_lessons` для user-course. | Есть только в полной витрине. |
| `video_visited_lessons` | Aggregated lesson feature | Число уроков, где `video_visited = True`. | Есть только в полной витрине. |
| `video_viewed_lessons` | Aggregated lesson feature | Число уроков, где `video_viewed = True`. | Есть только в полной витрине. |
| `translation_visited_lessons` | Aggregated lesson feature | Число уроков, где `translation_visited = True`. | Есть только в полной витрине. |
| `solved_lessons_count` | Aggregated lesson feature | Число уроков, отмеченных как solved. | Есть только в полной витрине. |
| `solved_tasks_total` | Aggregated lesson feature | Сумма `solved_tasks_count` по `user_lessons`. | Есть только в полной витрине. |
| `user_lessons_points_sum` | Aggregated lesson feature | Сумма `wk_points` по `user_lessons`. | Есть только в полной витрине. |
| `wk_solved_task_count_total` | Aggregated lesson feature | Сумма `wk_solved_task_count` по `user_lessons`. | Есть только в полной витрине. |
| `video_viewed_share_by_logged_lessons` | Aggregated lesson ratio | Доля просмотренных уроков среди уроков, где вообще есть лог. | Есть только в полной витрине. |
| `lesson_activity_events_total` | Aggregated event feature | Общее число событий из `user_activity_histories`, привязанных к user-course. | Полная история событий внутри уроков. |
| `lesson_activity_user_lessons_nunique` | Aggregated event feature | Число уникальных `user_lesson_id`, затронутых событиями. | Ширина активности внутри курса по полной истории. |
| `lesson_activity_days` | Aggregated event feature | Число календарных дней с lesson-activity. | Считается по полной истории `created_at`. |
| `lesson_activity_first_at` | Aggregated event feature | Первое событие `user_activity_histories` в курсе. | Минимум по полной истории `created_at`. |
| `lesson_activity_last_at` | Aggregated event feature | Последнее событие `user_activity_histories` в курсе. | Максимум по полной истории `created_at`. |
| `lesson_activity_count__show_conspect` | Aggregated event feature | Число событий `user_activity_histories` с action = `show_conspect`. | Полная история по курсу. |
| `lesson_activity_count__visit_translation` | Aggregated event feature | Число событий `user_activity_histories` с action = `visit_translation`. | Полная история по курсу. |
| `lesson_activity_count__visit_video` | Aggregated event feature | Число событий `user_activity_histories` с action = `visit_video`. | Полная история по курсу. |
| `actions_total` | Aggregated event feature | Общее число course-level действий из `wk_users_courses_actions`. | Полная история действий по курсу. |
| `action_days` | Aggregated event feature | Число календарных дней с course-level действиями. | Считается по полной истории `created_at`. |
| `action_first_at` | Aggregated event feature | Самое раннее действие в `wk_users_courses_actions`. | Минимум по полной истории `created_at`. |
| `action_last_at` | Aggregated event feature | Самое позднее действие в `wk_users_courses_actions`. | Максимум по полной истории `created_at`. |
| `action_count__scratch_playground_visited` | Aggregated event feature | Число событий `wk_users_courses_actions` с action = `scratch_playground_visited`. | Полная история по курсу. |
| `action_count__start_training` | Aggregated event feature | Число событий `wk_users_courses_actions` с action = `start_training`. | Полная история по курсу. |
| `action_count__user_answer` | Aggregated event feature | Число событий `wk_users_courses_actions` с action = `user_answer`. | Полная история по курсу. |
| `action_count__visit_preparation_material` | Aggregated event feature | Число событий `wk_users_courses_actions` с action = `visit_preparation_material`. | Полная история по курсу. |
| `action_count__visit_translation` | Aggregated event feature | Число событий `wk_users_courses_actions` с action = `visit_translation`. | Полная история по курсу. |
| `action_count__visit_video` | Aggregated event feature | Число событий `wk_users_courses_actions` с action = `visit_video`. | Полная история по курсу. |
| `media_sessions_total` | Aggregated media feature | Число media-сессий, привязанных к user-course. | Полная история media-сессий. |
| `media_resources_nunique` | Aggregated media feature | Число уникальных media-ресурсов в просмотре. | Разнообразие просмотренного контента по полной истории. |
| `media_watch_share_mean` | Aggregated media feature | Средняя доля просмотра media-сессий. | `watch_share` усредняется по сессиям. |
| `media_watch_share_max` | Aggregated media feature | Максимальная доля просмотра media-сессий. | Максимум по `watch_share`. |
| `media_sessions_80pct` | Aggregated media feature | Число media-сессий с `watch_share >= 0.8`. | Порог полного/достаточного просмотра. |
| `media_sessions_full` | Aggregated media feature | Число media-сессий с `watch_share >= 1.0`. | Фактически полные просмотры. |
| `media_started_at_first` | Aggregated media feature | Самый ранний старт media-сессии. | Минимум по `started_at`. |
| `media_started_at_last` | Aggregated media feature | Самый поздний старт media-сессии. | Максимум по `started_at`. |
| `trainings_rows` | Aggregated training feature | Число строк `user_trainings`, связанных с user-course. | Есть только в полной витрине. |
| `trainings_unique_total` | Aggregated training feature | Число уникальных trainings у пользователя в курсе. | Есть только в полной витрине. |
| `training_solved_tasks_total` | Aggregated training feature | Сумма решённых задач в trainings. | Полная история trainings. |
| `training_earned_points_sum` | Aggregated training feature | Сумма заработанных points в trainings. | Полная история trainings. |
| `training_submitted_answers_sum` | Aggregated training feature | Сумма отправленных ответов в trainings. | Полная история trainings. |
| `training_attempts_sum` | Aggregated training feature | Сумма попыток в trainings. | Полная история trainings. |
| `training_mark_max` | Aggregated training feature | Максимальная оценка по trainings. | Максимум по полной истории trainings. |
| `training_started_at_first` | Aggregated training feature | Самый ранний старт training в курсе. | Минимум по полной истории trainings. |
| `training_finished_at_last` | Aggregated training feature | Самое позднее завершение training. | Максимум по полной истории trainings. |
| `answer_rows` | Aggregated answer feature | Число строк `user_answers`, привязанных к user-course. | Полная история ответов по курсу. |
| `answer_tasks_nunique` | Aggregated answer feature | Число уникальных `task_id` в ответах. | Покрытие заданий ответами по полной истории. |
| `answer_solved_count` | Aggregated answer feature | Число ответов с `solved = True`. | Сумма solved по полной истории ответов. |
| `answer_skipped_count` | Aggregated answer feature | Число ответов с `skipped = True`. | Сумма skipped по полной истории ответов. |
| `answer_partial_count` | Aggregated answer feature | Число ответов с `wk_partial_answer = True`. | Частичные ответы по полной истории. |
| `answer_points_sum` | Aggregated answer feature | Сумма points по ответам. | Полная история course-level результатов по ответам. |
| `answer_attempts_sum` | Aggregated answer feature | Сумма attempts по ответам. | Полная история интенсивности попыток. |
| `answer_max_attempts_sum` | Aggregated answer feature | Сумма max_attempts по ответам. | Полная история доступных попыток. |
| `answer_async_nonzero_count` | Aggregated answer feature | Число ответов с ненулевым `async_check_status`. | Полная история асинхронной проверки. |
| `answer_first_submitted_at` | Aggregated answer feature | Самая ранняя отправка ответа. | Минимум по полной истории `submitted_at`. |
| `answer_last_submitted_at` | Aggregated answer feature | Самая поздняя отправка ответа. | Максимум по полной истории `submitted_at`. |

### Спецификация 2. Temporal-витрина `user_course_modeling_snapshots_long.csv`

| Параметр | Значение |
|---|---|
| Файл | `./artifacts/user_course_modeling_snapshots_long.csv` |
| Строк | `33604` |
| Столбцов | `69` |
| Гранулярность | 1 строка = 1 объект `module + user_id + course_id + cutoff_rule`. |
| Ключ | `module + user_id + course_id + cutoff_rule` |

Ключевые замечания:

- Это long-таблица snapshot-срезов: каждая пара `module + user_id + course_id` повторяется 4 раза с разными значениями `cutoff_rule`.
- Все event-признаки посчитаны только по событиям `<= cutoff_at`; поэтому `NaN` здесь часто означает не ошибку, а отсутствие события к моменту отсечения.
- Snapshot-витрина намеренно не включает `user_lessons`, user-profile state-поля и users-course state-поля, чтобы не тащить future leakage.
- `actual_target` и `heuristic_target` в snapshot-таблице — это reference labels, они копируются из population и не зависят от cutoff.

| Атрибут | Тип / роль | Смысл | Категории / важные примечания |
|---|---|---|---|
| `module` | Категория / часть ключа | Номер модуля в общей population. | Категории: {1, 2, 3, 4}. |
| `user_id` | ID / часть ключа | Идентификатор пользователя LMS. | Первичный user-level идентификатор. |
| `course_id` | ID / часть ключа | Идентификатор курса, к которому привязана строка. | Курс внутри LMS. |
| `cutoff_rule` | Категория / часть ключа | Правило temporal-отсечения, определяющее момент snapshot. | Категории: {7_days, 14_days, 21_days, midpoint}. |
| `club_name` | Категориальный population-атрибут | Текстовое название потока/клуба/группы из population-таблицы. | Высокая кардинальность: 141 значений. |
| `teacher_id` | ID / population-атрибут | Идентификатор преподавателя в population. | Можно использовать как cohort-like grouping id. |
| `cohort_id` | ID / population-атрибут | Идентификатор набора/кохорты внутри population. | Связан с group_template_id. |
| `level_bin` | Категориальный population-атрибут | Бинарный уровень программы из population. | Категории: {0, 1}. |
| `actual_target` | Label | Фактический бинарный таргет по известному статусу модуля. | Категории: {0.0, 1.0}; 1 = завершил, 0 = отчислен, NA = фактический статус неизвестен. Не использовать как feature. |
| `heuristic_target` | Proxy label | Эвристический бинарный таргет из `status_by_best_heuristic`. | Категории: {0.0, 1.0}; 1 = эвристически завершил, 0 = эвристически отчислен, NA = эвристики нет. Не использовать как feature. |
| `module_start_at` | Snapshot anchor | Восстановленная дата старта модуля для строки population. | Используется как точка отсчёта для всех cutoff. |
| `cutoff_at` | Snapshot anchor | Фактический момент отсечения, до которого разрешены события. | Все event-признаки в snapshot ограничены этой датой. |
| `course_video_duration_total` | Static course feature | Суммарная длительность видео по урокам курса. | При необходимости восстановлена через proxy из groups.duration. |
| `course_video_duration_mean` | Static course feature | Средняя длительность видео-урока в курсе. | При необходимости восстановлена через proxy из groups.duration. |
| `course_lesson_tasks_total` | Static course feature | Общее число lesson-task строк в курсе. | Структура курса, одинакова для всех пользователей курса. |
| `course_lesson_tasks_unique` | Static course feature | Число уникальных task_id в lesson_tasks курса. | Структура курса. |
| `course_required_tasks_total` | Static course feature | Число обязательных lesson-task в курсе. | Структура курса. |
| `access_periods_count` | Aggregated access feature | Число периодов доступа, привязанных к user-course. | В snapshot считаются только периоды, начавшиеся не позже cutoff. |
| `access_days_observed_total` | Aggregated access feature / snapshot | Суммарная наблюдаемая длительность доступа до `cutoff_at`. | Периоды обрезаются по cutoff. |
| `access_started_at_first` | Aggregated access feature | Самая ранняя дата начала доступа. | Минимум по всем access-периодам. |
| `access_observed_at_last` | Aggregated access feature / snapshot | Последний наблюдаемый момент доступа на `cutoff_at`. | Это `min(access_expired_at, cutoff_at)` в агрегате. |
| `lesson_activity_events_total` | Aggregated event feature | Общее число событий из `user_activity_histories`, привязанных к user-course. | Только события не позже cutoff. |
| `lesson_activity_user_lessons_nunique` | Aggregated event feature | Число уникальных `user_lesson_id`, затронутых событиями. | Только события не позже cutoff. |
| `lesson_activity_days` | Aggregated event feature | Число календарных дней с lesson-activity. | Считается только по `created_at <= cutoff_at`. |
| `lesson_activity_first_at` | Aggregated event feature | Первое событие `user_activity_histories` в курсе. | Минимум по событиям не позже cutoff. |
| `lesson_activity_last_at` | Aggregated event feature | Последнее событие `user_activity_histories` в курсе. | Максимум по событиям не позже cutoff. |
| `lesson_activity_count__show_conspect` | Aggregated event feature | Число событий `user_activity_histories` с action = `show_conspect`. | В snapshot считаются только события не позже cutoff. |
| `lesson_activity_count__visit_translation` | Aggregated event feature | Число событий `user_activity_histories` с action = `visit_translation`. | В snapshot считаются только события не позже cutoff. |
| `lesson_activity_count__visit_video` | Aggregated event feature | Число событий `user_activity_histories` с action = `visit_video`. | В snapshot считаются только события не позже cutoff. |
| `actions_total` | Aggregated event feature | Общее число course-level действий из `wk_users_courses_actions`. | Только действия не позже cutoff. |
| `action_days` | Aggregated event feature | Число календарных дней с course-level действиями. | Считается только по `created_at <= cutoff_at`. |
| `action_first_at` | Aggregated event feature | Самое раннее действие в `wk_users_courses_actions`. | Минимум по действиям не позже cutoff. |
| `action_last_at` | Aggregated event feature | Самое позднее действие в `wk_users_courses_actions`. | Максимум по действиям не позже cutoff. |
| `action_count__scratch_playground_visited` | Aggregated event feature | Число событий `wk_users_courses_actions` с action = `scratch_playground_visited`. | В snapshot считаются только события не позже cutoff. |
| `action_count__start_training` | Aggregated event feature | Число событий `wk_users_courses_actions` с action = `start_training`. | В snapshot считаются только события не позже cutoff. |
| `action_count__user_answer` | Aggregated event feature | Число событий `wk_users_courses_actions` с action = `user_answer`. | В snapshot считаются только события не позже cutoff. |
| `action_count__visit_preparation_material` | Aggregated event feature | Число событий `wk_users_courses_actions` с action = `visit_preparation_material`. | В snapshot считаются только события не позже cutoff. |
| `action_count__visit_translation` | Aggregated event feature | Число событий `wk_users_courses_actions` с action = `visit_translation`. | В snapshot считаются только события не позже cutoff. |
| `action_count__visit_video` | Aggregated event feature | Число событий `wk_users_courses_actions` с action = `visit_video`. | В snapshot считаются только события не позже cutoff. |
| `media_sessions_total` | Aggregated media feature | Число media-сессий, привязанных к user-course. | Только сессии, начавшиеся не позже cutoff. |
| `media_resources_nunique` | Aggregated media feature | Число уникальных media-ресурсов в просмотре. | Только ресурсы, встречающиеся до cutoff. |
| `media_watch_share_mean` | Aggregated media feature | Средняя доля просмотра media-сессий. | `watch_share` усредняется по сессиям. |
| `media_watch_share_max` | Aggregated media feature | Максимальная доля просмотра media-сессий. | Максимум по `watch_share`. |
| `media_sessions_80pct` | Aggregated media feature | Число media-сессий с `watch_share >= 0.8`. | Порог полного/достаточного просмотра. |
| `media_sessions_full` | Aggregated media feature | Число media-сессий с `watch_share >= 1.0`. | Фактически полные просмотры. |
| `media_started_at_first` | Aggregated media feature | Самый ранний старт media-сессии. | Минимум по `started_at`. |
| `media_started_at_last` | Aggregated media feature | Самый поздний старт media-сессии. | Максимум по `started_at`. |
| `trainings_started_total` | Aggregated training feature / snapshot | Число training-строк, которые уже стартовали к cutoff. | Ранний сигнал вовлечённости. |
| `trainings_started_nunique` | Aggregated training feature / snapshot | Число уникальных trainings, стартовавших к cutoff. | Snapshot-версия тренировочной активности. |
| `training_started_at_first` | Aggregated training feature | Самый ранний старт training в курсе. | Минимум по trainings, стартовавшим не позже cutoff. |
| `trainings_finished_total` | Aggregated training feature / snapshot | Число training-строк, завершённых к cutoff. | Outcome-часть training-активности. |
| `trainings_finished_nunique` | Aggregated training feature / snapshot | Число уникальных trainings, завершённых к cutoff. | Snapshot-версия завершённых trainings. |
| `training_solved_tasks_total` | Aggregated training feature | Сумма решённых задач в trainings. | Только по trainings, завершённым не позже cutoff. |
| `training_earned_points_sum` | Aggregated training feature | Сумма заработанных points в trainings. | Только по trainings, завершённым не позже cutoff. |
| `training_submitted_answers_sum` | Aggregated training feature | Сумма отправленных ответов в trainings. | Только по trainings, завершённым не позже cutoff. |
| `training_attempts_sum` | Aggregated training feature | Сумма попыток в trainings. | Только по trainings, завершённым не позже cutoff. |
| `training_mark_max` | Aggregated training feature | Максимальная оценка по trainings. | Только по trainings, завершённым не позже cutoff. |
| `training_finished_at_last` | Aggregated training feature | Самое позднее завершение training. | Максимум по trainings, завершённым не позже cutoff. |
| `answer_rows` | Aggregated answer feature | Число строк `user_answers`, привязанных к user-course. | Только ответы не позже cutoff. |
| `answer_tasks_nunique` | Aggregated answer feature | Число уникальных `task_id` в ответах. | Только задачи, по которым есть ответы до cutoff. |
| `answer_solved_count` | Aggregated answer feature | Число ответов с `solved = True`. | Только ответы не позже cutoff. |
| `answer_skipped_count` | Aggregated answer feature | Число ответов с `skipped = True`. | Только ответы не позже cutoff. |
| `answer_partial_count` | Aggregated answer feature | Число ответов с `wk_partial_answer = True`. | Только ответы не позже cutoff. |
| `answer_points_sum` | Aggregated answer feature | Сумма points по ответам. | Только ответы не позже cutoff. |
| `answer_attempts_sum` | Aggregated answer feature | Сумма attempts по ответам. | Только ответы не позже cutoff. |
| `answer_max_attempts_sum` | Aggregated answer feature | Сумма max_attempts по ответам. | Только ответы не позже cutoff. |
| `answer_async_nonzero_count` | Aggregated answer feature | Число ответов с ненулевым `async_check_status`. | Только ответы не позже cutoff. |
| `answer_first_submitted_at` | Aggregated answer feature | Самая ранняя отправка ответа. | Минимум по `submitted_at <= cutoff_at`. |
| `answer_last_submitted_at` | Aggregated answer feature | Самая поздняя отправка ответа. | Максимум по `submitted_at <= cutoff_at`. |